### LightGBM

In [3]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import json
import joblib
import copy
from pathlib import Path


In [4]:

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

LIGHTGBM_CORRECTED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "lightgbm_corrected"
)

LIGHTGBM_MODEL_DIR = (
    PROJECT_ROOT / "models" / "lightgbm"
)

LIGHTGBM_ARTIFACT_DIR = (
    PROJECT_ROOT / "artifacts" / "lightgbm"
)

LIGHTGBM_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LIGHTGBM_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LGB_FOLD_FILES = {}

for fold_number in range(1, 6):
    train_file = (
        LIGHTGBM_CORRECTED_DIR
        / f"fold_{fold_number}_train.parquet"
    )

    validation_file = (
        LIGHTGBM_CORRECTED_DIR
        / f"fold_{fold_number}_validation.parquet"
    )

    LGB_FOLD_FILES[fold_number] = {
        "train": train_file,
        "validation": validation_file
    }

    assert train_file.exists(), f"Missing file: {train_file}"
    assert validation_file.exists(), f"Missing file: {validation_file}"

print("Project root:", PROJECT_ROOT)
print("Corrected dataset directory:", LIGHTGBM_CORRECTED_DIR)
print("Model directory:", LIGHTGBM_MODEL_DIR)
print("Artifact directory:", LIGHTGBM_ARTIFACT_DIR)

print("\nDataset verification:")

for fold_number, files in LGB_FOLD_FILES.items():
    print(
        f"Fold {fold_number}: "
        f"training and validation files found"
    )

print("\nAll five folds are accessible.")

Project root: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence
Corrected dataset directory: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected
Model directory: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm
Artifact directory: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\artifacts\lightgbm

Dataset verification:
Fold 1: training and validation files found
Fold 2: training and validation files found
Fold 3: training and validation files found
Fold 4: training and validation files found
Fold 5: training and validation files found

All five folds are accessible.


In [5]:
SEED = 42

QUANTILES = [0.1, 0.5, 0.9]

LGB_CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "target_weekday",
    "target_event_name_1",
    "target_event_type_1"
]

LGB_NUMERICAL_FEATURES = [
    "sales_at_origin",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_std_7",
    "sales_rolling_mean_28",
    "sales_rolling_std_28",
    "sales_rolling_max_28",
    "origin_sell_price",
    "origin_price_missing",
    "origin_price_change",
    "origin_price_change_pct",
    "origin_price_vs_8w_avg",
    "origin_price_change_flag",
    "target_month",
    "target_year",
    "target_is_weekend",
    "target_day_of_month",
    "target_week_of_year",
    "target_is_event_day",
    "target_is_snap_day",
    "horizon"
]

MODEL_FEATURES_FINAL = (
    LGB_NUMERICAL_FEATURES
    + LGB_CATEGORICAL_FEATURES
)

TARGET_COLUMN = "target_sales"

LGB_PARAMS = {
    0.1: {
        "objective": "quantile",
        "alpha": 0.1,
        "metric": "quantile",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 100,
        "feature_fraction": 0.9,
        "verbosity": -1,
        "seed": SEED,
        "n_jobs": -1
    },
    0.5: {
        "objective": "quantile",
        "alpha": 0.5,
        "metric": "quantile",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 100,
        "feature_fraction": 0.9,
        "verbosity": -1,
        "seed": SEED,
        "n_jobs": -1
    },
    0.9: {
        "objective": "quantile",
        "alpha": 0.9,
        "metric": "quantile",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 100,
        "feature_fraction": 0.9,
        "verbosity": -1,
        "seed": SEED,
        "n_jobs": -1
    }
}

LGB_NUM_BOOST_ROUND = 3000
LGB_EARLY_STOPPING_ROUNDS = 100
LGB_LOG_PERIOD = 100

LGB_TRAINING_PARAMS = {
    quantile: {
        **copy.deepcopy(params),
        "num_threads": params["n_jobs"]
    }
    for quantile, params in LGB_PARAMS.items()
}

for params in LGB_TRAINING_PARAMS.values():
    params.pop("n_jobs")

print("LightGBM version:", lgb.__version__)
print("Quantiles:", QUANTILES)
print("Numerical features:", len(LGB_NUMERICAL_FEATURES))
print("Categorical features:", len(LGB_CATEGORICAL_FEATURES))
print("Total model features:", len(MODEL_FEATURES_FINAL))
print("Target:", TARGET_COLUMN)
print("Maximum boosting rounds:", LGB_NUM_BOOST_ROUND)
print("Early stopping rounds:", LGB_EARLY_STOPPING_ROUNDS)

assert len(MODEL_FEATURES_FINAL) == 32
assert len(set(MODEL_FEATURES_FINAL)) == 32
assert set(LGB_TRAINING_PARAMS) == set(QUANTILES)
assert all(
    params["objective"] == "quantile"
    for params in LGB_TRAINING_PARAMS.values()
)


LightGBM version: 4.7.0
Quantiles: [0.1, 0.5, 0.9]
Numerical features: 24
Categorical features: 8
Total model features: 32
Target: target_sales
Maximum boosting rounds: 3000
Early stopping rounds: 100


In [6]:
import duckdb
import pandas as pd

con = duckdb.connect()

expected_columns = (
    MODEL_FEATURES_FINAL
    + [TARGET_COLUMN]
)

audit_columns = [
    "id",
    "demand_class",
    "origin_day_num",
    "origin_day",
    "origin_type",
    "target_day_num",
    "target_day",
    "horizon"
]

required_columns = expected_columns + audit_columns

schema_results = []

for fold_number, files in LGB_FOLD_FILES.items():

    for dataset_type in ["train", "validation"]:

        file_path = files[dataset_type].as_posix()

        schema_df = con.execute(
            f"""
            DESCRIBE
            SELECT *
            FROM read_parquet('{file_path}')
            """
        ).df()

        actual_columns = schema_df["column_name"].tolist()

        missing_columns = [
            column
            for column in required_columns
            if column not in actual_columns
        ]

        extra_columns = [
            column
            for column in actual_columns
            if column not in required_columns
        ]

        row_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{file_path}')
            """
        ).fetchone()[0]

        schema_results.append({
            "fold": fold_number,
            "dataset": dataset_type,
            "rows": row_count,
            "columns": len(actual_columns),
            "missing_columns": missing_columns,
            "extra_columns": extra_columns
        })

con.close()

lgb_schema_audit_df = pd.DataFrame(schema_results)

display(lgb_schema_audit_df)

assert all(
    len(result["missing_columns"]) == 0
    for result in schema_results
), "One or more datasets are missing required columns."

assert all(
    len(result["extra_columns"]) == 0
    for result in schema_results
), "Unexpected columns found."

print("All five folds have the expected dataset schema.")

,fold,dataset,rows,columns,missing_columns,extra_columns
0,1,train,13776000,40,[],[]
1,1,validation,56000,40,[],[]
2,2,train,14056000,40,[],[]
3,2,validation,56000,40,[],[]
4,3,train,14336000,40,[],[]
5,3,validation,56000,40,[],[]
6,4,train,14616000,40,[],[]
7,4,validation,56000,40,[],[]
8,5,train,14896000,40,[],[]
9,5,validation,56000,40,[],[]


All five folds have the expected dataset schema.


In [7]:
import duckdb
import pandas as pd

con = duckdb.connect()

integrity_results = []

for fold_number, files in LGB_FOLD_FILES.items():
    for dataset_type in ["train", "validation"]:

        file_path = files[dataset_type].as_posix()

        numerical_checks = ",\n".join(
            [
                f"COUNT(*) FILTER (WHERE {feature} IS NULL) AS {feature}_nulls"
                for feature in LGB_NUMERICAL_FEATURES
            ]
            +
            [
                f"COUNT(*) FILTER (WHERE NOT isfinite(CAST({feature} AS DOUBLE))) AS {feature}_nonfinite"
                for feature in LGB_NUMERICAL_FEATURES
            ]
        )

        summary = con.execute(
            f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT id) AS series_count,
                COUNT(DISTINCT origin_day_num) AS origin_count,
                MIN(horizon) AS min_horizon,
                MAX(horizon) AS max_horizon,
                MIN(target_sales) AS min_target,
                MAX(target_sales) AS max_target,
                COUNT(*) FILTER (
                    WHERE target_sales IS NULL
                ) AS null_targets,
                COUNT(*) FILTER (
                    WHERE target_sales < 0
                ) AS negative_targets,
                COUNT(*) FILTER (
                    WHERE NOT isfinite(CAST(target_sales AS DOUBLE))
                ) AS nonfinite_targets,
                {numerical_checks}
            FROM read_parquet('{file_path}')
            """
        ).df()

        duplicate_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM (
                SELECT id, origin_day_num, horizon
                FROM read_parquet('{file_path}')
                GROUP BY id, origin_day_num, horizon
                HAVING COUNT(*) > 1
            )
            """
        ).fetchone()[0]

        result = summary.iloc[0].to_dict()

        result.update({
            "fold": fold_number,
            "dataset": dataset_type,
            "duplicate_keys": duplicate_count
        })

        integrity_results.append(result)

con.close()

lgb_integrity_audit_df = pd.DataFrame(integrity_results)

display(
    lgb_integrity_audit_df[
        [
            "fold",
            "dataset",
            "row_count",
            "series_count",
            "origin_count",
            "min_horizon",
            "max_horizon",
            "min_target",
            "max_target",
            "null_targets",
            "negative_targets",
            "nonfinite_targets",
            "duplicate_keys"
        ]
    ]
)

assert (lgb_integrity_audit_df["null_targets"] == 0).all()
assert (lgb_integrity_audit_df["negative_targets"] == 0).all()
assert (lgb_integrity_audit_df["nonfinite_targets"] == 0).all()
assert (lgb_integrity_audit_df["duplicate_keys"] == 0).all()
assert (lgb_integrity_audit_df["min_horizon"] == 1).all()
assert (lgb_integrity_audit_df["max_horizon"] == 28).all()

numerical_null_columns = [
    f"{feature}_nulls"
    for feature in LGB_NUMERICAL_FEATURES
]

numerical_nonfinite_columns = [
    f"{feature}_nonfinite"
    for feature in LGB_NUMERICAL_FEATURES
]

print("\nNumerical missing-value counts:")
display(
    lgb_integrity_audit_df[
        ["fold", "dataset"] + numerical_null_columns
    ]
)

assert (
    lgb_integrity_audit_df[numerical_nonfinite_columns] == 0
).all().all()

print("Dataset integrity audit passed.")

,fold,dataset,row_count,series_count,origin_count,min_horizon,max_horizon,min_target,max_target,null_targets,negative_targets,nonfinite_targets,duplicate_keys
0,1,train,13776000,2000,246,1,28,0,601,0,0,0,0
1,1,validation,56000,2000,1,1,28,0,126,0,0,0,0
2,2,train,14056000,2000,251,1,28,0,601,0,0,0,0
3,2,validation,56000,2000,1,1,28,0,145,0,0,0,0
4,3,train,14336000,2000,256,1,28,0,601,0,0,0,0
5,3,validation,56000,2000,1,1,28,0,114,0,0,0,0
6,4,train,14616000,2000,261,1,28,0,601,0,0,0,0
7,4,validation,56000,2000,1,1,28,0,98,0,0,0,0
8,5,train,14896000,2000,266,1,28,0,601,0,0,0,0
9,5,validation,56000,2000,1,1,28,0,179,0,0,0,0



Numerical missing-value counts:


,fold,dataset,sales_at_origin_nulls,sales_lag_1_nulls,sales_lag_7_nulls,sales_lag_14_nulls,sales_lag_28_nulls,sales_rolling_mean_7_nulls,sales_rolling_std_7_nulls,sales_rolling_mean_28_nulls,...,origin_price_vs_8w_avg_nulls,origin_price_change_flag_nulls,target_month_nulls,target_year_nulls,target_is_weekend_nulls,target_day_of_month_nulls,target_week_of_year_nulls,target_is_event_day_nulls,target_is_snap_day_nulls,horizon_nulls
0,1,train,0,0,0,0,0,0,0,0,...,3197068,3197068,0,0,0,0,0,0,0,0
1,1,validation,0,0,0,0,0,0,0,0,...,448,448,0,0,0,0,0,0,0,0
2,2,train,0,0,0,0,0,0,0,0,...,3199308,3199308,0,0,0,0,0,0,0,0
3,2,validation,0,0,0,0,0,0,0,0,...,308,308,0,0,0,0,0,0,0,0
4,3,train,0,0,0,0,0,0,0,0,...,3201296,3201296,0,0,0,0,0,0,0,0
5,3,validation,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,4,train,0,0,0,0,0,0,0,0,...,3202052,3202052,0,0,0,0,0,0,0,0
7,4,validation,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,5,train,0,0,0,0,0,0,0,0,...,3202052,3202052,0,0,0,0,0,0,0,0
9,5,validation,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Dataset integrity audit passed.


In [8]:
import duckdb
import joblib
from pathlib import Path

LGB_ENCODED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "lightgbm_encoded"
)

LGB_ENCODED_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

LGB_CATEGORY_MAPPINGS = {}
LGB_ENCODED_FILES = {}

for fold_number, files in LGB_FOLD_FILES.items():

    print(f"\nProcessing fold {fold_number}")

    train_path = files["train"].as_posix()

    fold_mappings = {}

    for feature in LGB_CATEGORICAL_FEATURES:

        mapping_table = f"cat_map_{feature}"

        con.execute(
            f"""
            CREATE OR REPLACE TEMP TABLE {mapping_table} AS
            SELECT
                category_value,
                CAST(
                    ROW_NUMBER() OVER (
                        ORDER BY category_value
                    ) - 1
                    AS INTEGER
                ) AS category_code
            FROM (
                SELECT DISTINCT
                    CAST({feature} AS VARCHAR) AS category_value
                FROM read_parquet('{train_path}')
                WHERE {feature} IS NOT NULL
            )
            """
        )

        mapping_rows = con.execute(
            f"""
            SELECT category_value, category_code
            FROM {mapping_table}
            ORDER BY category_code
            """
        ).fetchall()

        fold_mappings[feature] = {
            category_value: category_code
            for category_value, category_code in mapping_rows
        }

        print(
            f"{feature}: "
            f"{len(mapping_rows)} training categories"
        )

    LGB_CATEGORY_MAPPINGS[fold_number] = fold_mappings

    encoded_fold_files = {}

    for dataset_type in ["train", "validation"]:

        source_path = files[dataset_type].as_posix()

        output_path = (
            LGB_ENCODED_DIR
            / f"fold_{fold_number}_{dataset_type}_encoded.parquet"
        )

        source_path_sql = source_path.replace("'", "''")
        output_path_sql = output_path.as_posix().replace("'", "''")

        joins = []
        encoded_columns = []

        for feature in LGB_CATEGORICAL_FEATURES:

            mapping_table = f"cat_map_{feature}"
            mapping_alias = f"map_{feature}"

            joins.append(
                f"""
                LEFT JOIN {mapping_table} AS {mapping_alias}
                    ON CAST(src.{feature} AS VARCHAR)
                    = {mapping_alias}.category_value
                """
            )

            encoded_columns.append(
                f"""
                COALESCE(
                    {mapping_alias}.category_code,
                    -1
                )::INTEGER AS {feature}
                """
            )

        join_sql = "\n".join(joins)
        encoded_sql = ",\n".join(encoded_columns)

        excluded_columns = ", ".join(LGB_CATEGORICAL_FEATURES)

        transformation_query = f"""
            SELECT
                src.* EXCLUDE ({excluded_columns}),
                {encoded_sql}
            FROM read_parquet('{source_path_sql}') AS src
            {join_sql}
        """

        con.execute(
            f"""
            COPY (
                {transformation_query}
            )
            TO '{output_path_sql}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
        )

        encoded_fold_files[dataset_type] = output_path

        print(
            f"Saved {dataset_type}: {output_path.name}"
        )

        unknown_checks = []

        for feature in LGB_CATEGORICAL_FEATURES:
            mapping_alias = f"map_{feature}"

            unknown_checks.append(
                f"""
                COUNT(*) FILTER (
                    WHERE src.{feature} IS NOT NULL
                    AND {mapping_alias}.category_code IS NULL
                ) AS {feature}_unseen
                """
            )

        unknown_query = f"""
            SELECT
                {", ".join(unknown_checks)}
            FROM read_parquet('{source_path_sql}') AS src
            {join_sql}
        """

        unknown_result = con.execute(unknown_query).df()

        unseen_count = int(
            unknown_result.sum(axis=1).iloc[0]
        )

        print("Unseen non-null category observations:", unseen_count)

        assert unseen_count == 0, (
            f"Unseen categories found in fold {fold_number}, "
            f"{dataset_type}"
        )

    LGB_ENCODED_FILES[fold_number] = encoded_fold_files

con.close()

LGB_CATEGORY_MAPPING_FILE = (
    LGB_ENCODED_DIR / "category_mappings.joblib"
)

joblib.dump(
    LGB_CATEGORY_MAPPINGS,
    LGB_CATEGORY_MAPPING_FILE
)

print("\nAll fold-specific category mappings saved.")
print("Mapping file:", LGB_CATEGORY_MAPPING_FILE)


Processing fold 1
item_id: 1490 training categories
dept_id: 7 training categories
cat_id: 3 training categories
store_id: 10 training categories
state_id: 3 training categories
target_weekday: 7 training categories
target_event_name_1: 31 training categories
target_event_type_1: 5 training categories
Saved train: fold_1_train_encoded.parquet
Unseen non-null category observations: 0
Saved validation: fold_1_validation_encoded.parquet
Unseen non-null category observations: 0

Processing fold 2
item_id: 1490 training categories
dept_id: 7 training categories
cat_id: 3 training categories
store_id: 10 training categories
state_id: 3 training categories
target_weekday: 7 training categories
target_event_name_1: 31 training categories
target_event_type_1: 5 training categories
Saved train: fold_2_train_encoded.parquet
Unseen non-null category observations: 0
Saved validation: fold_2_validation_encoded.parquet
Unseen non-null category observations: 0

Processing fold 3
item_id: 1490 trainin

In [9]:
import duckdb
import pandas as pd

con = duckdb.connect()

encoded_audit_results = []

for fold_number, files in LGB_ENCODED_FILES.items():

    for dataset_type, encoded_path in files.items():

        file_path = encoded_path.as_posix()

        categorical_checks = ",\n".join(
            [
                f"MIN({feature}) AS {feature}_min_code"
                for feature in LGB_CATEGORICAL_FEATURES
            ]
            +
            [
                f"MAX({feature}) AS {feature}_max_code"
                for feature in LGB_CATEGORICAL_FEATURES
            ]
            +
            [
                f"""
                COUNT(*) FILTER (
                    WHERE {feature} < -1
                ) AS {feature}_invalid_codes
                """
                for feature in LGB_CATEGORICAL_FEATURES
            ]
        )

        numerical_checks = ",\n".join(
            [
                f"""
                COUNT(*) FILTER (
                    WHERE NOT isfinite(CAST({feature} AS DOUBLE))
                ) AS {feature}_nonfinite
                """
                for feature in LGB_NUMERICAL_FEATURES
            ]
        )

        result = con.execute(
            f"""
            SELECT
                COUNT(*) AS row_count,
                COUNT(DISTINCT id) AS series_count,
                COUNT(*) FILTER (
                    WHERE target_sales IS NULL
                ) AS null_targets,
                COUNT(*) FILTER (
                    WHERE target_sales < 0
                ) AS negative_targets,
                {categorical_checks},
                {numerical_checks}
            FROM read_parquet('{file_path}')
            """
        ).df()

        original_path = (
            LGB_FOLD_FILES[fold_number][dataset_type].as_posix()
        )

        original_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{original_path}')
            """
        ).fetchone()[0]

        result["fold"] = fold_number
        result["dataset"] = dataset_type
        result["original_row_count"] = original_count

        encoded_audit_results.append(result.iloc[0].to_dict())

con.close()

lgb_encoded_audit_df = pd.DataFrame(encoded_audit_results)

display(
    lgb_encoded_audit_df[
        [
            "fold",
            "dataset",
            "original_row_count",
            "row_count",
            "series_count",
            "null_targets",
            "negative_targets"
        ]
    ]
)

assert (
    lgb_encoded_audit_df["row_count"]
    == lgb_encoded_audit_df["original_row_count"]
).all()

assert (lgb_encoded_audit_df["null_targets"] == 0).all()
assert (lgb_encoded_audit_df["negative_targets"] == 0).all()

invalid_code_columns = [
    f"{feature}_invalid_codes"
    for feature in LGB_CATEGORICAL_FEATURES
]

nonfinite_columns = [
    f"{feature}_nonfinite"
    for feature in LGB_NUMERICAL_FEATURES
]

assert (
    lgb_encoded_audit_df[invalid_code_columns] == 0
).all().all()

assert (
    lgb_encoded_audit_df[nonfinite_columns] == 0
).all().all()

print("Encoded dataset integrity verification passed.")

,fold,dataset,original_row_count,row_count,series_count,null_targets,negative_targets
0,1,train,13776000,13776000,2000,0,0
1,1,validation,56000,56000,2000,0,0
2,2,train,14056000,14056000,2000,0,0
3,2,validation,56000,56000,2000,0,0
4,3,train,14336000,14336000,2000,0,0
5,3,validation,56000,56000,2000,0,0
6,4,train,14616000,14616000,2000,0,0
7,4,validation,56000,56000,2000,0,0
8,5,train,14896000,14896000,2000,0,0
9,5,validation,56000,56000,2000,0,0


Encoded dataset integrity verification passed.


In [10]:
import duckdb
import pandas as pd

con = duckdb.connect()

expected_encoded_columns = (
    MODEL_FEATURES_FINAL
    + [TARGET_COLUMN]
    + [
        "id",
        "demand_class",
        "origin_day_num",
        "origin_day",
        "origin_type",
        "target_day_num",
        "target_day"
    ]
)

schema_results = []

for fold_number, files in LGB_ENCODED_FILES.items():

    for dataset_type, encoded_path in files.items():

        file_path = encoded_path.as_posix()

        schema_df = con.execute(
            f"""
            DESCRIBE
            SELECT *
            FROM read_parquet('{file_path}')
            """
        ).df()

        actual_columns = schema_df["column_name"].tolist()

        missing_columns = [
            column
            for column in expected_encoded_columns
            if column not in actual_columns
        ]

        unexpected_columns = [
            column
            for column in actual_columns
            if column not in expected_encoded_columns
        ]

        schema_results.append({
            "fold": fold_number,
            "dataset": dataset_type,
            "column_count": len(actual_columns),
            "missing_columns": missing_columns,
            "unexpected_columns": unexpected_columns
        })

        for feature in LGB_CATEGORICAL_FEATURES:
            feature_type = schema_df.loc[
                schema_df["column_name"] == feature,
                "column_type"
            ].iloc[0]

            assert "INT" in feature_type.upper(), (
                f"{feature} is not integer encoded: {feature_type}"
            )

con.close()

lgb_encoded_schema_df = pd.DataFrame(schema_results)

display(lgb_encoded_schema_df)

assert all(
    len(result["missing_columns"]) == 0
    and len(result["unexpected_columns"]) == 0
    for result in schema_results
)

print("All encoded datasets have the expected schema.")

,fold,dataset,column_count,missing_columns,unexpected_columns
0,1,train,40,[],[]
1,1,validation,40,[],[]
2,2,train,40,[],[]
3,2,validation,40,[],[]
4,3,train,40,[],[]
5,3,validation,40,[],[]
6,4,train,40,[],[]
7,4,validation,40,[],[]
8,5,train,40,[],[]
9,5,validation,40,[],[]


All encoded datasets have the expected schema.


In [11]:
LGB_MODEL_INPUT_COLUMNS = (
    LGB_NUMERICAL_FEATURES
    + LGB_CATEGORICAL_FEATURES
)

LGB_TRAINING_COLUMNS = (
    LGB_MODEL_INPUT_COLUMNS
    + [TARGET_COLUMN]
)

LGB_METADATA_COLUMNS = [
    "id",
    "demand_class",
    "origin_day_num",
    "origin_day",
    "origin_type",
    "target_day_num",
    "target_day"
]

LGB_DATASET_FILES = {
    fold_number: {
        "train": files["train"],
        "validation": files["validation"]
    }
    for fold_number, files in LGB_ENCODED_FILES.items()
}

assert len(LGB_MODEL_INPUT_COLUMNS) == 32
assert len(LGB_TRAINING_COLUMNS) == 33
assert len(set(LGB_MODEL_INPUT_COLUMNS)) == 32
assert set(LGB_MODEL_INPUT_COLUMNS).isdisjoint(
    LGB_METADATA_COLUMNS
)

print("Model input features:", len(LGB_MODEL_INPUT_COLUMNS))
print("Training columns including target:", len(LGB_TRAINING_COLUMNS))
print("Categorical features:", LGB_CATEGORICAL_FEATURES)
print("Metadata columns excluded from model:", LGB_METADATA_COLUMNS)

print("\nPrepared fold files:")

for fold_number, files in LGB_DATASET_FILES.items():
    print(
        f"Fold {fold_number}: "
        f"train={files['train'].name}, "
        f"validation={files['validation'].name}"
    )

print("\nLightGBM data preparation completed.")

Model input features: 32
Training columns including target: 33
Categorical features: ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'target_weekday', 'target_event_name_1', 'target_event_type_1']
Metadata columns excluded from model: ['id', 'demand_class', 'origin_day_num', 'origin_day', 'origin_type', 'target_day_num', 'target_day']

Prepared fold files:
Fold 1: train=fold_1_train_encoded.parquet, validation=fold_1_validation_encoded.parquet
Fold 2: train=fold_2_train_encoded.parquet, validation=fold_2_validation_encoded.parquet
Fold 3: train=fold_3_train_encoded.parquet, validation=fold_3_validation_encoded.parquet
Fold 4: train=fold_4_train_encoded.parquet, validation=fold_4_validation_encoded.parquet
Fold 5: train=fold_5_train_encoded.parquet, validation=fold_5_validation_encoded.parquet

LightGBM data preparation completed.


#### Load Development Fold 1

In [13]:
import pandas as pd
import numpy as np

fold_number = 1

train_path = LGB_DATASET_FILES[fold_number]["train"]
validation_path = LGB_DATASET_FILES[fold_number]["validation"]

LGB_NUMERICAL_DTYPES = {
    feature: "float32"
    for feature in LGB_NUMERICAL_FEATURES
}

LGB_CATEGORICAL_DTYPES = {
    feature: "int32"
    for feature in LGB_CATEGORICAL_FEATURES
}

LGB_COLUMN_DTYPES = {
    **LGB_NUMERICAL_DTYPES,
    **LGB_CATEGORICAL_DTYPES,
    TARGET_COLUMN: "float32"
}

lgb_fold1_train = pd.read_parquet(
    train_path,
    columns=LGB_TRAINING_COLUMNS
)

lgb_fold1_validation = pd.read_parquet(
    validation_path,
    columns=LGB_TRAINING_COLUMNS
)

for feature, dtype in LGB_COLUMN_DTYPES.items():
    lgb_fold1_train[feature] = (
        lgb_fold1_train[feature].astype(dtype)
    )

    lgb_fold1_validation[feature] = (
        lgb_fold1_validation[feature].astype(dtype)
    )

X_lgb_fold1_train = lgb_fold1_train[LGB_MODEL_INPUT_COLUMNS]
y_lgb_fold1_train = lgb_fold1_train[TARGET_COLUMN]

X_lgb_fold1_validation = (
    lgb_fold1_validation[LGB_MODEL_INPUT_COLUMNS]
)

y_lgb_fold1_validation = (
    lgb_fold1_validation[TARGET_COLUMN]
)

print("Training shape:", X_lgb_fold1_train.shape)
print("Validation shape:", X_lgb_fold1_validation.shape)

print(
    "Training memory (GB):",
    round(
        lgb_fold1_train.memory_usage(deep=True).sum()
        / 1024**3,
        3
    )
)

print(
    "Validation memory (GB):",
    round(
        lgb_fold1_validation.memory_usage(deep=True).sum()
        / 1024**3,
        3
    )
)

print("\nCategorical data types:")
print(X_lgb_fold1_train[LGB_CATEGORICAL_FEATURES].dtypes)

print("\nTarget summary:")
display(y_lgb_fold1_train.describe())

assert X_lgb_fold1_train.shape[1] == 32
assert X_lgb_fold1_validation.shape[1] == 32
assert len(X_lgb_fold1_train) == len(y_lgb_fold1_train)
assert len(X_lgb_fold1_validation) == len(y_lgb_fold1_validation)

assert all(
    X_lgb_fold1_train[feature].dtype == np.int32
    for feature in LGB_CATEGORICAL_FEATURES
)


Training shape: (13776000, 32)
Validation shape: (56000, 32)
Training memory (GB): 1.694
Validation memory (GB): 0.007

Categorical data types:
item_id                int32
dept_id                int32
cat_id                 int32
store_id               int32
state_id               int32
target_weekday         int32
target_event_name_1    int32
target_event_type_1    int32
dtype: object

Target summary:


count    1.377600e+07
mean     1.079023e+00
std      3.696692e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      6.010000e+02
Name: target_sales, dtype: float64

In [14]:
lgb_fold1_train_dataset = lgb.Dataset(
    data=X_lgb_fold1_train,
    label=y_lgb_fold1_train,
    feature_name=LGB_MODEL_INPUT_COLUMNS,
    categorical_feature=LGB_CATEGORICAL_FEATURES,
    free_raw_data=True
)

lgb_fold1_validation_dataset = lgb.Dataset(
    data=X_lgb_fold1_validation,
    label=y_lgb_fold1_validation,
    reference=lgb_fold1_train_dataset,
    feature_name=LGB_MODEL_INPUT_COLUMNS,
    categorical_feature=LGB_CATEGORICAL_FEATURES,
    free_raw_data=True
)

lgb_fold1_train_dataset.construct()
lgb_fold1_validation_dataset.construct()

print("Training dataset constructed.")
print("Validation dataset constructed.")

print("\nTraining dataset:")
print("Rows:", lgb_fold1_train_dataset.num_data())
print("Features:", lgb_fold1_train_dataset.num_feature())

print("\nValidation dataset:")
print("Rows:", lgb_fold1_validation_dataset.num_data())
print("Features:", lgb_fold1_validation_dataset.num_feature())

print("\nCategorical features:")
print(lgb_fold1_train_dataset.get_feature_name())

assert lgb_fold1_train_dataset.num_data() == 13_776_000
assert lgb_fold1_validation_dataset.num_data() == 56_000
assert lgb_fold1_train_dataset.num_feature() == 32
assert lgb_fold1_validation_dataset.num_feature() == 32

print("\nLightGBM datasets constructed and validated.")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
Training dataset constructed.
Validation dataset constructed.

Training dataset:
Rows: 13776000
Features: 32

Validation dataset:
Rows: 56000
Features: 32

Categorical features:
['sales_at_origin', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_std_7', 'sales_rolling_mean_28', 'sales_rolling_std_28', 'sales_rolling_max_28', 'origin_sell_price', 'origin_price_missing', 'origin_price_change', 'origin_price_change_pct', 'origin_price_vs_8w_avg', 'origin_price_change_flag', 'target_month', 'target_year', 'target_is_weekend', 'target_day_of_month', 'target_week_of_year', 'target_is_event_day', 'target_is_snap_day', 'horizon', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'target_weekday', 'target_even

In [15]:
import time

LGB_FOLD1_MODELS = {}
LGB_FOLD1_RESULTS = {}

def train_lgb_quantile_model(
    quantile,
    train_dataset,
    validation_dataset,
    params,
    num_boost_round=3000,
    early_stopping_rounds=100,
    log_period=100
):
    print(f"\nTraining LightGBM quantile model: {quantile}")

    model_params = copy.deepcopy(params)

    start_time = time.time()

    model = lgb.train(
        params=model_params,
        train_set=train_dataset,
        num_boost_round=num_boost_round,
        valid_sets=[validation_dataset],
        valid_names=["validation"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=early_stopping_rounds,
                first_metric_only=True,
                verbose=True
            ),
            lgb.log_evaluation(period=log_period)
        ]
    )

    elapsed_time = time.time() - start_time

    best_iteration = model.best_iteration
    best_score = model.best_score["validation"]["quantile"]

    result = {
        "quantile": quantile,
        "best_iteration": best_iteration,
        "best_validation_quantile_loss": best_score,
        "training_time_seconds": elapsed_time,
        "training_time_minutes": elapsed_time / 60
    }

    print(f"\nQuantile: {quantile}")
    print("Best iteration:", best_iteration)
    print("Best validation quantile loss:", best_score)
    print("Training time (minutes):", round(elapsed_time / 60, 2))

    return model, result


print("Quantile training function defined.")

Quantile training function defined.


In [17]:
lgb_fold1_p50_model, lgb_fold1_p50_result = train_lgb_quantile_model(
    quantile=0.5,
    train_dataset=lgb_fold1_train_dataset,
    validation_dataset=lgb_fold1_validation_dataset,
    params=LGB_TRAINING_PARAMS[0.5],
    num_boost_round=LGB_NUM_BOOST_ROUND,
    early_stopping_rounds=LGB_EARLY_STOPPING_ROUNDS,
    log_period=LGB_LOG_PERIOD
)

LGB_FOLD1_MODELS[0.5] = lgb_fold1_p50_model
LGB_FOLD1_RESULTS[0.5] = lgb_fold1_p50_result

print("\nP50 model training completed.")


Training LightGBM quantile model: 0.5
Training until validation scores don't improve for 100 rounds
[100]	validation's quantile: 0.452877
[200]	validation's quantile: 0.452388
[300]	validation's quantile: 0.45205
[400]	validation's quantile: 0.45202
[500]	validation's quantile: 0.451845
[600]	validation's quantile: 0.45183
[700]	validation's quantile: 0.45179
[800]	validation's quantile: 0.451778
[900]	validation's quantile: 0.451646
[1000]	validation's quantile: 0.451664
Early stopping, best iteration is:
[911]	validation's quantile: 0.451646
Evaluated only: quantile

Quantile: 0.5
Best iteration: 911
Best validation quantile loss: 0.4516462414328664
Training time (minutes): 7.65

P50 model training completed.


In [ ]:
from pathlib import Path
import json
import joblib
import shutil
import zipfile
import numpy as np
import pandas as pd
import lightgbm as lgb

LGB_FOLD_MODELS = {
    1: {
        0.5: lgb_fold1_p50_model
    }
}

LGB_FOLD_RESULTS = {
    1: {
        0.5: lgb_fold1_p50_result
    }
}

LGB_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "lightgbm_output"

LGB_MODEL_DIR = LGB_OUTPUT_DIR / "models"
LGB_ARTIFACT_DIR = LGB_OUTPUT_DIR / "artifacts"
LGB_RESULTS_DIR = LGB_OUTPUT_DIR / "results"

for directory in [
    LGB_MODEL_DIR,
    LGB_ARTIFACT_DIR,
    LGB_RESULTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("LightGBM output directories initialised.")
print("Output directory:", LGB_OUTPUT_DIR)

# LGB_OUTPUT_DIR = PROJECT_ROOT / "models" / "lightgbm"

# LGB_MODEL_DIR = LGB_OUTPUT_DIR / "models"
# LGB_ARTIFACT_DIR = LGB_OUTPUT_DIR / "artifacts"
# LGB_RESULTS_DIR = LGB_OUTPUT_DIR / "results"

# for directory in [
#     LGB_MODEL_DIR,
#     LGB_ARTIFACT_DIR,
#     LGB_RESULTS_DIR
# ]:
#     directory.mkdir(parents=True, exist_ok=True)

# print("Output directory:", LGB_OUTPUT_DIR)

LightGBM output directories initialised.
Output directory: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_output


In [20]:
for fold_number in [1, 2, 3]:

    print(f"\n{'=' * 60}")
    print(f"Training LightGBM Fold {fold_number}")
    print(f"{'=' * 60}")

    if fold_number not in LGB_FOLD_MODELS:
        LGB_FOLD_MODELS[fold_number] = {}

    if fold_number not in LGB_FOLD_RESULTS:
        LGB_FOLD_RESULTS[fold_number] = {}

    train_path = LGB_DATASET_FILES[fold_number]["train"]
    validation_path = LGB_DATASET_FILES[fold_number]["validation"]

    if fold_number == 1:
        train_dataset = lgb_fold1_train_dataset
        validation_dataset = lgb_fold1_validation_dataset

    else:
        train_df = pd.read_parquet(
            train_path,
            columns=LGB_TRAINING_COLUMNS
        )

        validation_df = pd.read_parquet(
            validation_path,
            columns=LGB_TRAINING_COLUMNS
        )

        for feature in LGB_NUMERICAL_FEATURES:
            train_df[feature] = train_df[feature].astype("float32")
            validation_df[feature] = validation_df[feature].astype("float32")

        for feature in LGB_CATEGORICAL_FEATURES:
            train_df[feature] = train_df[feature].astype("int32")
            validation_df[feature] = validation_df[feature].astype("int32")

        X_train = train_df[MODEL_FEATURES_FINAL]
        y_train = train_df[TARGET_COLUMN]

        X_validation = validation_df[MODEL_FEATURES_FINAL]
        y_validation = validation_df[TARGET_COLUMN]

        train_dataset = lgb.Dataset(
            data=X_train,
            label=y_train,
            feature_name=MODEL_FEATURES_FINAL,
            categorical_feature=LGB_CATEGORICAL_FEATURES,
            free_raw_data=True
        )

        validation_dataset = lgb.Dataset(
            data=X_validation,
            label=y_validation,
            reference=train_dataset,
            feature_name=MODEL_FEATURES_FINAL,
            categorical_feature=LGB_CATEGORICAL_FEATURES,
            free_raw_data=True
        )

        train_dataset.construct()
        validation_dataset.construct()

    for quantile in QUANTILES:

        if quantile in LGB_FOLD_MODELS[fold_number]:
            print(
                f"Fold {fold_number} P{int(quantile * 100)} "
                "already trained. Skipping."
            )
            continue

        model, result = train_lgb_quantile_model(
            quantile=quantile,
            train_dataset=train_dataset,
            validation_dataset=validation_dataset,
            params=LGB_TRAINING_PARAMS[quantile],
            num_boost_round=LGB_NUM_BOOST_ROUND,
            early_stopping_rounds=LGB_EARLY_STOPPING_ROUNDS,
            log_period=LGB_LOG_PERIOD
        )

        LGB_FOLD_MODELS[fold_number][quantile] = model
        LGB_FOLD_RESULTS[fold_number][quantile] = result

        print(
            f"Fold {fold_number} "
            f"P{int(quantile * 100)} training completed."
        )

    if fold_number != 1:
        del train_df, validation_df
        del X_train, y_train, X_validation, y_validation

    del train_dataset, validation_dataset

    import gc
    gc.collect()

print("\nAll development fold training completed.")


Training LightGBM Fold 1

Training LightGBM quantile model: 0.1
Training until validation scores don't improve for 100 rounds
[100]	validation's quantile: 0.121833
[200]	validation's quantile: 0.121829
Early stopping, best iteration is:
[110]	validation's quantile: 0.121829
Evaluated only: quantile

Quantile: 0.1
Best iteration: 110
Best validation quantile loss: 0.12182853471468433
Training time (minutes): 1.88
Fold 1 P10 training completed.
Fold 1 P50 already trained. Skipping.

Training LightGBM quantile model: 0.9
Training until validation scores don't improve for 100 rounds
[100]	validation's quantile: 0.348776
[200]	validation's quantile: 0.347973
Early stopping, best iteration is:
[173]	validation's quantile: 0.347907
Evaluated only: quantile

Quantile: 0.9
Best iteration: 173
Best validation quantile loss: 0.3479066296085273
Training time (minutes): 1.92
Fold 1 P90 training completed.

Training LightGBM Fold 2

Training LightGBM quantile model: 0.1
Training until validation sc

In [22]:
import numpy as np
import pandas as pd
import lightgbm as lgb

assert set(LGB_FOLD_MODELS.keys()) == {1, 2, 3}
assert set(LGB_FOLD_RESULTS.keys()) == {1, 2, 3}

LGB_TRAINING_SUMMARY = []

for fold_number in [1, 2, 3]:

    for quantile in QUANTILES:

        model = LGB_FOLD_MODELS[fold_number][quantile]
        result = LGB_FOLD_RESULTS[fold_number][quantile]

        assert isinstance(model, lgb.Booster)
        assert model.best_iteration > 0
        assert result["best_iteration"] == model.best_iteration
        assert np.isfinite(
            result["best_validation_quantile_loss"]
        )

        LGB_TRAINING_SUMMARY.append({
            "fold": fold_number,
            "quantile": quantile,
            "best_iteration": model.best_iteration,
            "validation_quantile_loss": (
                result["best_validation_quantile_loss"]
            ),
            "training_time_minutes": (
                result["training_time_minutes"]
            )
        })

lgb_training_summary_df = pd.DataFrame(
    LGB_TRAINING_SUMMARY
).sort_values(
    ["fold", "quantile"]
).reset_index(drop=True)

display(lgb_training_summary_df)

assert len(lgb_training_summary_df) == 9


,fold,quantile,best_iteration,validation_quantile_loss,training_time_minutes
0,1,0.1,110,0.121829,1.878922
1,1,0.5,911,0.451646,7.651674
2,1,0.9,173,0.347907,1.921998
3,2,0.1,140,0.130653,2.129202
4,2,0.5,175,0.465970,2.467938
5,2,0.9,613,0.316535,4.655142
6,3,0.1,60,0.128421,1.121755
7,3,0.5,172,0.446290,2.037768
8,3,0.9,380,0.309283,3.189578


In [23]:
from pathlib import Path
import lightgbm as lgb

PROJECT_ROOT = Path(
    r"C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence"
)

LGB_OUTPUT_DIR = PROJECT_ROOT / "models" / "lightgbm"

LGB_MODEL_DIR = LGB_OUTPUT_DIR / "models"
LGB_ARTIFACT_DIR = LGB_OUTPUT_DIR / "artifacts"
LGB_RESULTS_DIR = LGB_OUTPUT_DIR / "results"

for directory in [
    LGB_MODEL_DIR,
    LGB_ARTIFACT_DIR,
    LGB_RESULTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

for fold_number in [1, 2, 3]:

    for quantile in QUANTILES:

        model = LGB_FOLD_MODELS[fold_number][quantile]

        assert isinstance(model, lgb.Booster)

        model_path = (
            LGB_MODEL_DIR
            / f"lightgbm_fold_{fold_number}_p{int(quantile * 100)}.txt"
        )

        model.save_model(
            str(model_path),
            num_iteration=model.best_iteration
        )

        assert model_path.exists()
        assert model_path.stat().st_size > 0

        print("Saved:", model_path)

print("\nAll nine LightGBM models saved successfully.")

Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_1_p10.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_1_p50.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_1_p90.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_2_p10.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_2_p50.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_2_p90.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_3_p10.txt
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\models\lightgbm_fold_3_p50.txt
Saved: C

#### Save Training Result

In [24]:
import json
import joblib
import shutil
import zipfile
import gc
import numpy as np
import pandas as pd
import lightgbm as lgb

LGB_TRAINING_SUMMARY = []

for fold_number in [1, 2, 3]:

    for quantile in QUANTILES:

        result = LGB_FOLD_RESULTS[fold_number][quantile]

        LGB_TRAINING_SUMMARY.append({
            "fold": fold_number,
            "quantile": quantile,
            "best_iteration": result["best_iteration"],
            "validation_quantile_loss": (
                result["best_validation_quantile_loss"]
            ),
            "training_time_minutes": (
                result["training_time_minutes"]
            )
        })

lgb_training_summary_df = pd.DataFrame(
    LGB_TRAINING_SUMMARY
).sort_values(
    ["fold", "quantile"]
).reset_index(drop=True)

display(lgb_training_summary_df)

assert len(lgb_training_summary_df) == 9

summary_path = (
    LGB_RESULTS_DIR
    / "lightgbm_development_training_summary.csv"
)

lgb_training_summary_df.to_csv(
    summary_path,
    index=False
)

LGB_DETAILED_RESULTS = {}

for fold_number in [1, 2, 3]:

    LGB_DETAILED_RESULTS[str(fold_number)] = {}

    for quantile in QUANTILES:

        result = LGB_FOLD_RESULTS[fold_number][quantile]

        LGB_DETAILED_RESULTS[str(fold_number)][str(quantile)] = {
            key: (
                value.item()
                if isinstance(value, np.generic)
                else value
            )
            for key, value in result.items()
        }

detailed_results_path = (
    LGB_RESULTS_DIR
    / "lightgbm_development_training_results.json"
)

with open(detailed_results_path, "w", encoding="utf-8") as file:
    json.dump(
        LGB_DETAILED_RESULTS,
        file,
        indent=4
    )

print("Training summary saved:", summary_path)
print("Detailed results saved:", detailed_results_path)

,fold,quantile,best_iteration,validation_quantile_loss,training_time_minutes
0,1,0.1,110,0.121829,1.878922
1,1,0.5,911,0.451646,7.651674
2,1,0.9,173,0.347907,1.921998
3,2,0.1,140,0.130653,2.129202
4,2,0.5,175,0.465970,2.467938
5,2,0.9,613,0.316535,4.655142
6,3,0.1,60,0.128421,1.121755
7,3,0.5,172,0.446290,2.037768
8,3,0.9,380,0.309283,3.189578


Training summary saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\results\lightgbm_development_training_summary.csv
Detailed results saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\results\lightgbm_development_training_results.json


#### Save model configuration

In [25]:
LGB_MODEL_CONFIG = {
    "model_name": "LightGBM Quantile Regression",
    "project": "AI-Powered Retail Demand Intelligence Platform",
    "dataset": "M5 Forecasting Accuracy",
    "forecast_horizon": 28,
    "quantiles": QUANTILES,
    "target_column": TARGET_COLUMN,
    "model_features": MODEL_FEATURES_FINAL,
    "numerical_features": LGB_NUMERICAL_FEATURES,
    "categorical_features": LGB_CATEGORICAL_FEATURES,
    "random_seed": SEED,
    "training_folds": [1, 2, 3],
    "training_parameters": {
        str(quantile): dict(params)
        for quantile, params in LGB_PARAMS.items()
    },
    "num_boost_round": LGB_NUM_BOOST_ROUND,
    "early_stopping_rounds": LGB_EARLY_STOPPING_ROUNDS,
    "log_period": LGB_LOG_PERIOD
}

config_path = (
    LGB_ARTIFACT_DIR
    / "lightgbm_model_config.json"
)

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        LGB_MODEL_CONFIG,
        file,
        indent=4
    )

print("Model configuration saved:", config_path)

Model configuration saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\artifacts\lightgbm_model_config.json


#### Save categorical mappings

In [26]:
LGB_CATEGORY_MAPPING_PATH = (
    LGB_ENCODED_DIR / "category_mappings.joblib"
)

assert LGB_CATEGORY_MAPPING_PATH.exists(), (
    f"Categorical mapping file not found: "
    f"{LGB_CATEGORY_MAPPING_PATH}"
)

saved_mapping_path = (
    LGB_ARTIFACT_DIR / "category_mappings.joblib"
)

shutil.copy2(
    LGB_CATEGORY_MAPPING_PATH,
    saved_mapping_path
)

assert saved_mapping_path.exists()

print("Categorical mappings saved:", saved_mapping_path)

Categorical mappings saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\artifacts\category_mappings.joblib


#### Generate validation predictions

In [27]:
LGB_VALIDATION_PREDICTIONS = {}

for fold_number in [1, 2, 3]:

    print(f"\nGenerating predictions for Fold {fold_number}")

    validation_path = LGB_DATASET_FILES[fold_number]["validation"]

    validation_columns = list(dict.fromkeys(
        LGB_METADATA_COLUMNS
        + [TARGET_COLUMN]
        + MODEL_FEATURES_FINAL
    ))

    validation_df = pd.read_parquet(
        validation_path,
        columns=validation_columns
    )

    X_validation = validation_df[MODEL_FEATURES_FINAL].copy()

    for feature in LGB_NUMERICAL_FEATURES:
        X_validation[feature] = X_validation[feature].astype("float32")

    for feature in LGB_CATEGORICAL_FEATURES:
        X_validation[feature] = X_validation[feature].astype("int32")

    prediction_columns = list(dict.fromkeys(
        LGB_METADATA_COLUMNS + [TARGET_COLUMN]
    ))

    prediction_df = validation_df[prediction_columns].copy()

    for quantile in QUANTILES:

        model = LGB_FOLD_MODELS[fold_number][quantile]

        predictions = model.predict(
            X_validation,
            num_iteration=model.best_iteration
        )

        prediction_column = (
            f"prediction_p{int(quantile * 100)}"
        )

        prediction_df[prediction_column] = (
            predictions.astype("float32")
        )

        assert len(predictions) == len(validation_df)
        assert np.isfinite(predictions).all()

    prediction_path = (
        LGB_RESULTS_DIR
        / f"lightgbm_fold_{fold_number}_validation_predictions.parquet"
    )

    prediction_df.to_parquet(
        prediction_path,
        index=False
    )

    LGB_VALIDATION_PREDICTIONS[fold_number] = prediction_df

    print("Validation rows:", len(prediction_df))
    print("Prediction columns:", [
        f"prediction_p{int(q * 100)}"
        for q in QUANTILES
    ])
    print("Saved:", prediction_path.name)

    del validation_df, X_validation, prediction_df
    gc.collect()

print("\nAll three folds' validation predictions saved.")


Generating predictions for Fold 1
Validation rows: 56000
Prediction columns: ['prediction_p10', 'prediction_p50', 'prediction_p90']
Saved: lightgbm_fold_1_validation_predictions.parquet

Generating predictions for Fold 2
Validation rows: 56000
Prediction columns: ['prediction_p10', 'prediction_p50', 'prediction_p90']
Saved: lightgbm_fold_2_validation_predictions.parquet

Generating predictions for Fold 3
Validation rows: 56000
Prediction columns: ['prediction_p10', 'prediction_p50', 'prediction_p90']
Saved: lightgbm_fold_3_validation_predictions.parquet

All three folds' validation predictions saved.


#### Evaluate quantile crossing
- Quantile crossing occurs when the predicted quantiles violate their expected ordering:

In [28]:
LGB_QUANTILE_CROSSING_RESULTS = []

for fold_number in [1, 2, 3]:

    prediction_df = LGB_VALIDATION_PREDICTIONS[fold_number]

    p10 = prediction_df["prediction_p10"].to_numpy()
    p50 = prediction_df["prediction_p50"].to_numpy()
    p90 = prediction_df["prediction_p90"].to_numpy()

    crossing_p10_p50 = p10 > p50
    crossing_p50_p90 = p50 > p90
    crossing_p10_p90 = p10 > p90

    any_crossing = (
        crossing_p10_p50 | crossing_p50_p90
    )

    LGB_QUANTILE_CROSSING_RESULTS.append({
        "fold": fold_number,
        "validation_rows": len(prediction_df),
        "p10_above_p50": int(crossing_p10_p50.sum()),
        "p50_above_p90": int(crossing_p50_p90.sum()),
        "p10_above_p90": int(crossing_p10_p90.sum()),
        "any_crossing": int(any_crossing.sum()),
        "crossing_rate": float(any_crossing.mean())
    })

lgb_crossing_df = pd.DataFrame(
    LGB_QUANTILE_CROSSING_RESULTS
)

display(lgb_crossing_df)

crossing_path = (
    LGB_RESULTS_DIR
    / "lightgbm_quantile_crossing_results.csv"
)

lgb_crossing_df.to_csv(
    crossing_path,
    index=False
)

print("Quantile crossing results saved:", crossing_path)

,fold,validation_rows,p10_above_p50,p50_above_p90,p10_above_p90,any_crossing,crossing_rate
0,1,56000,962,281,281,1243,0.022196
1,2,56000,232,308,308,540,0.009643
2,3,56000,515,245,230,760,0.013571


Quantile crossing results saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\results\lightgbm_quantile_crossing_results.csv


#### Verify saved models can be reloaded

In [29]:
LGB_MODEL_VERIFICATION = []

for fold_number in [1, 2, 3]:

    for quantile in QUANTILES:

        model_path = (
            LGB_MODEL_DIR
            / f"lightgbm_fold_{fold_number}_p{int(quantile * 100)}.txt"
        )

        assert model_path.exists()
        assert model_path.stat().st_size > 0

        loaded_model = lgb.Booster(
            model_file=str(model_path)
        )

        original_model = LGB_FOLD_MODELS[fold_number][quantile]

        assert (
            loaded_model.num_feature()
            == original_model.num_feature()
        )

        assert (
            loaded_model.current_iteration()
            == original_model.best_iteration
        )

        LGB_MODEL_VERIFICATION.append({
            "fold": fold_number,
            "quantile": quantile,
            "model_file": model_path.name,
            "features": loaded_model.num_feature(),
            "iterations": loaded_model.current_iteration(),
            "file_size_mb": (
                model_path.stat().st_size / (1024 ** 2)
            ),
            "status": "PASSED"
        })

        del loaded_model

lgb_model_verification_df = pd.DataFrame(
    LGB_MODEL_VERIFICATION
)

display(lgb_model_verification_df)

assert len(lgb_model_verification_df) == 9
assert (
    lgb_model_verification_df["status"] == "PASSED"
).all()

print("All nine saved models passed reload verification.")

,fold,quantile,model_file,features,iterations,file_size_mb,status
0,1,0.1,lightgbm_fold_1_p10.txt,32,110,0.324060,PASSED
1,1,0.5,lightgbm_fold_1_p50.txt,32,911,3.006996,PASSED
2,1,0.9,lightgbm_fold_1_p90.txt,32,173,0.934695,PASSED
3,2,0.1,lightgbm_fold_2_p10.txt,32,140,0.414879,PASSED
4,2,0.5,lightgbm_fold_2_p50.txt,32,175,0.587043,PASSED
5,2,0.9,lightgbm_fold_2_p90.txt,32,613,2.992439,PASSED
6,3,0.1,lightgbm_fold_3_p10.txt,32,60,0.176797,PASSED
7,3,0.5,lightgbm_fold_3_p50.txt,32,172,0.573243,PASSED
8,3,0.9,lightgbm_fold_3_p90.txt,32,380,1.948152,PASSED


All nine saved models passed reload verification.


#### Create final ZIP package

In [30]:
DOWNLOAD_PACKAGE = (
    PROJECT_ROOT
    / "models"
    / "lightgbm_development_outputs.zip"
)

with zipfile.ZipFile(
    DOWNLOAD_PACKAGE,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zip_file:

    for directory in [
        LGB_MODEL_DIR,
        LGB_ARTIFACT_DIR,
        LGB_RESULTS_DIR
    ]:

        for file_path in directory.rglob("*"):

            if file_path.is_file():

                archive_path = file_path.relative_to(
                    LGB_OUTPUT_DIR
                )

                zip_file.write(
                    file_path,
                    arcname=str(archive_path)
                )

assert DOWNLOAD_PACKAGE.exists()
assert DOWNLOAD_PACKAGE.stat().st_size > 0

print("ZIP package created successfully.")
print("Location:", DOWNLOAD_PACKAGE)
print(
    "Package size (MB):",
    round(DOWNLOAD_PACKAGE.stat().st_size / (1024 ** 2), 2)
)

ZIP package created successfully.
Location: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm_development_outputs.zip
Package size (MB): 3.3


In [31]:
LGB_OUTPUT_INVENTORY = []

for directory in [
    LGB_MODEL_DIR,
    LGB_ARTIFACT_DIR,
    LGB_RESULTS_DIR
]:

    for file_path in sorted(directory.rglob("*")):

        if file_path.is_file():

            LGB_OUTPUT_INVENTORY.append({
                "category": directory.name,
                "file": file_path.name,
                "size_mb": round(
                    file_path.stat().st_size / (1024 ** 2),
                    3
                ),
                "status": "SAVED"
            })

lgb_output_inventory_df = pd.DataFrame(
    LGB_OUTPUT_INVENTORY
)

display(lgb_output_inventory_df)

print("\nModel files:", len(list(LGB_MODEL_DIR.glob("*.txt"))))
print("Artifact files:", len(list(LGB_ARTIFACT_DIR.iterdir())))
print("Result files:", len(list(LGB_RESULTS_DIR.iterdir())))
print("ZIP exists:", DOWNLOAD_PACKAGE.exists())

,category,file,size_mb,status
0,models,lightgbm_fold_1_p10.txt,0.324,SAVED
1,models,lightgbm_fold_1_p50.txt,3.007,SAVED
2,models,lightgbm_fold_1_p90.txt,0.935,SAVED
3,models,lightgbm_fold_2_p10.txt,0.415,SAVED
4,models,lightgbm_fold_2_p50.txt,0.587,SAVED
5,models,lightgbm_fold_2_p90.txt,2.992,SAVED
6,models,lightgbm_fold_3_p10.txt,0.177,SAVED
7,models,lightgbm_fold_3_p50.txt,0.573,SAVED
8,models,lightgbm_fold_3_p90.txt,1.948,SAVED
9,artifacts,category_mappings.joblib,0.137,SAVED



Model files: 9
Artifact files: 2
Result files: 6
ZIP exists: True


#### Model Evaluation

## 1. Point Forecast Accuracy

| Metric | What it measures | Interpretation |
| :--- | :--- | :--- |
| **MAE (Mean Absolute Error)** | Average absolute difference between actual and predicted sales. | Lower is better. Easy to interpret in units of sales. |
| **RMSE (Root Mean Squared Error)** | Square root of the average squared prediction errors. | Lower is better. Penalises large errors more heavily than MAE. |
| **WAPE (Weighted Absolute Percentage Error)** | Total absolute error divided by total actual sales, expressed as a percentage. | Lower is better. Measures overall error relative to total demand. |

## 2. Probabilistic Forecast Accuracy

| Metric | What it measures | Interpretation |
| :--- | :--- | :--- |
| **Pinball Loss** | Measures the accuracy of a predicted quantile (P10, P50 or P90), penalising under- and over-predictions asymmetrically. | Lower is better. Evaluates whether each quantile forecast is useful. |
| **P10 Coverage** | Percentage of actual sales that are less than or equal to the P10 forecast. | Ideally around 10%. |
| **P90 Coverage** | Percentage of actual sales that are less than or equal to the P90 forecast. | Ideally around 90%. |
| **Prediction Interval Coverage** | Percentage of actual sales falling between P10 and P90. | Ideally around 80%, indicating the interval captures the expected proportion of actual outcomes. |
| **Prediction Interval Width** | Average distance between P90 and P10 forecasts. | Narrower is generally better, provided coverage remains adequate. |

## 3. Forecast Reliability

| Metric | What it measures | Interpretation |
| :--- | :--- | :--- |
| **Quantile Crossing Rate** | Percentage of predictions where quantiles violate their expected order, such as P10 being greater than P50 or P50 being greater than P90. | Ideally 0%. Crossing means the predicted quantiles are internally inconsistent. |

## Important Interpretation Notes

* **MAE, RMSE and WAPE** evaluate P50 forecasts in this project.
* **Pinball Loss** evaluates each quantile separately; compare the same quantile across folds, not P10 against P50 directly.
* **Coverage** measures calibration, not forecast accuracy by itself. A model can achieve 80% interval coverage simply by producing excessively wide intervals.
* **Interval Width and Coverage** should be assessed together. The goal is reasonably narrow intervals with coverage close to 80%.

In [33]:
LGB_EVALUATION_RESULTS = []

for fold_number in [1, 2, 3]:

    prediction_df = LGB_VALIDATION_PREDICTIONS[fold_number]

    actual = prediction_df[TARGET_COLUMN].to_numpy(dtype=np.float64)

    p10 = prediction_df["prediction_p10"].to_numpy(dtype=np.float64)
    p50 = prediction_df["prediction_p50"].to_numpy(dtype=np.float64)
    p90 = prediction_df["prediction_p90"].to_numpy(dtype=np.float64)

    assert len(actual) == len(p10) == len(p50) == len(p90)
    assert np.isfinite(actual).all()
    assert np.isfinite(p10).all()
    assert np.isfinite(p50).all()
    assert np.isfinite(p90).all()
    assert (actual >= 0).all()

    errors = actual - p50

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))

    total_actual = np.sum(actual)

    assert total_actual > 0

    wape = (
        np.sum(np.abs(errors)) / total_actual
    ) * 100

    pinball_losses = {}

    for quantile, predictions in [
        (0.1, p10),
        (0.5, p50),
        (0.9, p90)
    ]:

        quantile_errors = actual - predictions

        pinball_loss = np.mean(
            np.maximum(
                quantile * quantile_errors,
                (quantile - 1) * quantile_errors
            )
        )

        pinball_losses[quantile] = pinball_loss

    p10_coverage = np.mean(actual <= p10)
    p90_coverage = np.mean(actual <= p90)

    interval_coverage = np.mean(
        (actual >= p10) & (actual <= p90)
    )

    interval_width = np.mean(p90 - p10)

    crossing_rate = np.mean(
        (p10 > p50) | (p50 > p90)
    )

    LGB_EVALUATION_RESULTS.append({
        "fold": fold_number,
        "validation_rows": len(actual),
        "MAE_P50": mae,
        "RMSE_P50": rmse,
        "WAPE_P50_percent": wape,
        "pinball_P10": pinball_losses[0.1],
        "pinball_P50": pinball_losses[0.5],
        "pinball_P90": pinball_losses[0.9],
        "P10_empirical_coverage": p10_coverage,
        "P90_empirical_coverage": p90_coverage,
        "P10_P90_interval_coverage": interval_coverage,
        "P10_P90_mean_interval_width": interval_width,
        "quantile_crossing_rate": crossing_rate
    })

lgb_evaluation_df = pd.DataFrame(
    LGB_EVALUATION_RESULTS
)

display(
    lgb_evaluation_df.round(6)
)

assert len(lgb_evaluation_df) == 3
assert np.isfinite(
    lgb_evaluation_df.select_dtypes(include=[np.number]).to_numpy()
).all()


,fold,validation_rows,MAE_P50,RMSE_P50,WAPE_P50_percent,pinball_P10,pinball_P50,pinball_P90,P10_empirical_coverage,P90_empirical_coverage,P10_P90_interval_coverage,P10_P90_mean_interval_width,quantile_crossing_rate
0,1,56000,0.903292,2.709361,72.735138,0.121829,0.451646,0.347907,0.608893,0.893268,0.886179,2.649179,0.022196
1,2,56000,0.931940,2.403668,69.934541,0.130653,0.465970,0.316535,0.591554,0.888143,0.880554,2.850178,0.009643
2,3,56000,0.892580,2.251032,67.696626,0.128421,0.446290,0.309283,0.584429,0.899786,0.895357,2.996572,0.013571


In [40]:
LGB_P10_DIAGNOSTICS = []

for fold_number in [1, 2, 3]:

    prediction_df = LGB_VALIDATION_PREDICTIONS[fold_number]

    actual = prediction_df["target_sales"].to_numpy(dtype=np.float64)
    p10 = prediction_df["prediction_p10"].to_numpy(dtype=np.float64)
    p50 = prediction_df["prediction_p50"].to_numpy(dtype=np.float64)
    p90 = prediction_df["prediction_p90"].to_numpy(dtype=np.float64)

    zero_actual = actual == 0
    positive_actual = actual > 0

    diagnostics = {
        "fold": fold_number,
        "actual_zero_rate": np.mean(zero_actual),
        "actual_positive_rate": np.mean(positive_actual),

        "p10_zero_or_negative_rate": np.mean(p10 <= 0),
        "p10_positive_rate": np.mean(p10 > 0),
        "p10_mean": np.mean(p10),
        "p10_median": np.median(p10),

        "p10_coverage_actual_zero": (
            np.mean(actual[zero_actual] <= p10[zero_actual])
            if np.any(zero_actual) else np.nan
        ),
        "p10_coverage_actual_positive": (
            np.mean(actual[positive_actual] <= p10[positive_actual])
            if np.any(positive_actual) else np.nan
        ),

        "p50_zero_or_negative_rate": np.mean(p50 <= 0),
        "p90_zero_or_negative_rate": np.mean(p90 <= 0),

        "p10_p50_crossing_rate": np.mean(p10 > p50),
        "p50_p90_crossing_rate": np.mean(p50 > p90),
    }

    LGB_P10_DIAGNOSTICS.append(diagnostics)

lgb_p10_diagnostics_df = pd.DataFrame(LGB_P10_DIAGNOSTICS)

display(lgb_p10_diagnostics_df.round(6))

,fold,actual_zero_rate,actual_positive_rate,p10_zero_or_negative_rate,p10_positive_rate,p10_mean,p10_median,p10_coverage_actual_zero,p10_coverage_actual_positive,p50_zero_or_negative_rate,p90_zero_or_negative_rate,p10_p50_crossing_rate,p50_p90_crossing_rate
0,1,0.609893,0.390107,0.9355,0.0645,0.052422,0.0,0.996633,0.002701,0.551911,0.005018,0.017179,0.005018
1,2,0.591911,0.408089,0.9310,0.0690,0.061666,0.0,0.997375,0.002932,0.554250,0.005500,0.004143,0.005500
2,3,0.583946,0.416054,0.9400,0.0600,0.053562,0.0,0.999327,0.002103,0.524732,0.004107,0.009196,0.004375


In [45]:
LGB_DEMAND_CLASS_EVALUATION = []

for fold_number in [1, 2, 3]:

    prediction_df = LGB_VALIDATION_PREDICTIONS[fold_number].copy()

    for demand_class, group in prediction_df.groupby("demand_class"):

        actual = group["target_sales"].to_numpy(dtype=np.float64)
        p10 = group["prediction_p10"].to_numpy(dtype=np.float64)
        p50 = group["prediction_p50"].to_numpy(dtype=np.float64)
        p90 = group["prediction_p90"].to_numpy(dtype=np.float64)

        errors = actual - p50

        p10_strict_coverage = np.mean(actual < p10)
        p10_inclusive_coverage = np.mean(actual <= p10)

        p90_strict_coverage = np.mean(actual < p90)
        p90_inclusive_coverage = np.mean(actual <= p90)

        LGB_DEMAND_CLASS_EVALUATION.append({
            "fold": fold_number,
            "demand_class": demand_class,
            "rows": len(group),

            "actual_zero_rate": np.mean(actual == 0),

            "MAE_P50": np.mean(np.abs(errors)),
            "RMSE_P50": np.sqrt(np.mean(errors ** 2)),

            "P10_zero_prediction_rate": np.mean(p10 <= 0),
            "P10_strict_coverage": p10_strict_coverage,
            "P10_inclusive_coverage": p10_inclusive_coverage,

            "P90_strict_coverage": p90_strict_coverage,
            "P90_inclusive_coverage": p90_inclusive_coverage,

            "P10_P90_interval_coverage": np.mean(
                (actual >= p10) & (actual <= p90)
            ),

            "mean_interval_width": np.mean(p90 - p10),

            "quantile_crossing_rate": np.mean(
                (p10 > p50) | (p50 > p90)
            )
        })

lgb_demand_class_df = pd.DataFrame(LGB_DEMAND_CLASS_EVALUATION)

display(
    lgb_demand_class_df
    .sort_values(["fold", "demand_class"])
    .round(6)
    .reset_index(drop=True)
)

,fold,demand_class,rows,actual_zero_rate,MAE_P50,RMSE_P50,P10_zero_prediction_rate,P10_strict_coverage,P10_inclusive_coverage,P90_strict_coverage,P90_inclusive_coverage,P10_P90_interval_coverage,mean_interval_width,quantile_crossing_rate
0,1,erratic,1624,0.157020,3.385730,5.730964,0.379310,0.067734,0.156404,0.897783,0.897783,0.830049,11.554111,0.000000
1,1,intermittent,40320,0.679439,0.580036,2.198472,0.988194,0.000719,0.679365,0.897222,0.897222,0.896503,1.718415,0.023636
2,1,lumpy,10024,0.534816,1.255280,2.618248,0.941341,0.009876,0.526237,0.878292,0.878292,0.868416,3.371803,0.027534
3,1,no_demand,448,0.930804,0.122768,0.532430,1.000000,0.000000,0.930804,0.930804,0.930804,0.930804,0.000435,0.000000
4,1,smooth,3584,0.202567,2.528176,5.121716,0.570312,0.044364,0.212054,0.883929,0.883929,0.839565,7.395235,0.003906
5,2,erratic,1624,0.198892,3.513471,5.882230,0.465517,0.095443,0.213670,0.899015,0.899015,0.803571,11.642083,0.000000
6,2,intermittent,40320,0.660020,0.575466,1.448547,0.982639,0.001513,0.660094,0.891840,0.891840,0.890327,1.841457,0.010739
7,2,lumpy,10024,0.519952,1.387226,3.054711,0.935754,0.007881,0.512271,0.879888,0.879888,0.872007,3.704194,0.010275
8,2,no_demand,448,0.638393,0.705933,1.352487,1.000000,0.000000,0.638393,0.747768,0.747768,0.747768,0.641180,0.000000
9,2,smooth,3584,0.199219,2.527394,4.966304,0.539062,0.036272,0.207589,0.882254,0.882254,0.845982,8.102003,0.001116


### LightGBM Quantile Regression — Training Findings

#### Training Summary
- Trained three quantile regression models (P10, P50 and P90) across three chronological development folds using LightGBM.
- Used early stopping with a maximum of 3,000 boosting rounds and 100 early-stopping rounds.
- All nine models completed training successfully.
- Best iterations varied across folds and quantiles, indicating different convergence behaviour.

#### Validation Findings
- P50 achieved MAE of **0.89–0.93** and WAPE of **67.70%–72.74%** across the three folds.
- Performance was reasonably consistent across folds, although RMSE varied from **2.25 to 2.71**.
- P90 empirical coverage ranged from **88.81% to 89.98%**, close to the nominal 90% quantile.
- P10 empirical coverage ranged from **58.44% to 60.89%**, primarily reflecting the high proportion of zero-sales observations and the discrete, zero-inflated demand distribution.
- P10–P90 interval coverage ranged from **88.06% to 89.54%**, exceeding the nominal 80% coverage.
- Quantile crossing remained relatively low, between **0.96% and 2.22%**.

#### Demand-Class Findings
- **Intermittent demand:** Lowest P50 MAE (approximately 0.57–0.58), partly influenced by its high zero-sales frequency.
- **Erratic demand:** Highest P50 MAE (approximately 3.04–3.51) and widest prediction intervals.
- **Lumpy demand:** Moderate forecasting errors, with some quantile crossing.
- **Smooth demand:** More challenging than intermittent demand, with P50 MAE around 2.40–2.53.
- **No-demand:** Future sales were not consistently zero despite historical classification, highlighting the limitations of static demand classification.

#### Overall Conclusion
LightGBM successfully produced point and probabilistic demand forecasts across all three development folds. Its P90 forecasts showed relatively good empirical calibration, while its lower quantile and prediction intervals require careful interpretation due to zero-inflated demand. Forecasting performance also varied considerably by demand class.

**Model selection remains pending:** LightGBM must be compared against Seasonal Naive on identical validation observations before its forecasting value can be established. Further evaluation should also examine performance by forecast horizon and demand class.

### Seasonal Naive Baseline - Comparison with LightGBM

Seasonal naïve is a fundamental baseline forecasting method where future predictions equal the actual value from the same period in the previous seasonal cycle.

#### Quick Summary
* **Rule:** If sales on a given day last week (or year) were $y$, the forecast for today is $y$.
* **Formula:** $\hat{y}_{T+h\vert{}T} = y_{T+h-m}$ (where $m$ is the seasonal period, such as $m=7$ for daily data with weekly seasonality).
* **Best Use:** Highly seasonal data with stable patterns and no strong upward/downward trend.
* **Primary Value:** Serves as the primary benchmark baseline. Advanced models (like LightGBM, LSTM, and TFT) must reliably outperform it to justify their complexity.

In [46]:
from pathlib import Path

SEASONAL_NAIVE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "seasonal_naive"
    / "seasonal_naive_evaluation.parquet"
)

seasonal_naive_df = pd.read_parquet(SEASONAL_NAIVE_PATH)

FOLD_EVALUATION_TYPES = {
    1: "development_fold_1",
    2: "development_fold_2",
    3: "development_fold_3"
}

LGB_NAIVE_COMPARISON = []
LGB_NAIVE_CLASS_COMPARISON = []

comparison_keys = [
    "id",
    "origin_day_num",
    "target_day_num"
]

for fold_number in [1, 2, 3]:

    print(f"\nAuditing Fold {fold_number}")

    lgb_df = LGB_VALIDATION_PREDICTIONS[fold_number].copy()

    naive_df = seasonal_naive_df.loc[
        seasonal_naive_df["evaluation_type"]
        == FOLD_EVALUATION_TYPES[fold_number]
    ].copy()

    naive_df = naive_df[
        comparison_keys
        + ["target_sales", "naive_forecast"]
    ]

    assert not lgb_df.duplicated(comparison_keys).any()
    assert not naive_df.duplicated(comparison_keys).any()

    comparison_df = lgb_df.merge(
        naive_df,
        on=comparison_keys,
        how="inner",
        suffixes=("_lgb", "_naive"),
        validate="one_to_one"
    )

    assert len(comparison_df) == len(lgb_df)
    assert len(comparison_df) == len(naive_df)

    actual_lgb = comparison_df["target_sales_lgb"].to_numpy(
        dtype=np.float64
    )
    actual_naive = comparison_df["target_sales_naive"].to_numpy(
        dtype=np.float64
    )

    assert np.array_equal(actual_lgb, actual_naive)

    actual = actual_lgb
    lgb_prediction = comparison_df["prediction_p50"].to_numpy(
        dtype=np.float64
    )
    naive_prediction = comparison_df["naive_forecast"].to_numpy(
        dtype=np.float64
    )

    assert np.isfinite(actual).all()
    assert np.isfinite(lgb_prediction).all()
    assert np.isfinite(naive_prediction).all()

    lgb_error = actual - lgb_prediction
    naive_error = actual - naive_prediction

    lgb_mae = np.mean(np.abs(lgb_error))
    naive_mae = np.mean(np.abs(naive_error))

    lgb_rmse = np.sqrt(np.mean(lgb_error ** 2))
    naive_rmse = np.sqrt(np.mean(naive_error ** 2))

    actual_total = np.sum(actual)

    assert actual_total > 0

    lgb_wape = np.sum(np.abs(lgb_error)) / actual_total * 100
    naive_wape = np.sum(np.abs(naive_error)) / actual_total * 100

    LGB_NAIVE_COMPARISON.append({
        "fold": fold_number,
        "rows": len(comparison_df),

        "LightGBM_MAE": lgb_mae,
        "Seasonal_Naive_MAE": naive_mae,
        "MAE_improvement_percent": (
            (naive_mae - lgb_mae) / naive_mae * 100
        ),

        "LightGBM_RMSE": lgb_rmse,
        "Seasonal_Naive_RMSE": naive_rmse,
        "RMSE_improvement_percent": (
            (naive_rmse - lgb_rmse) / naive_rmse * 100
        ),

        "LightGBM_WAPE_percent": lgb_wape,
        "Seasonal_Naive_WAPE_percent": naive_wape,
        "WAPE_improvement_percent": (
            (naive_wape - lgb_wape) / naive_wape * 100
        )
    })

    for demand_class, group in comparison_df.groupby("demand_class"):

        actual_class = group["target_sales_lgb"].to_numpy(
            dtype=np.float64
        )
        lgb_class_prediction = group["prediction_p50"].to_numpy(
            dtype=np.float64
        )
        naive_class_prediction = group["naive_forecast"].to_numpy(
            dtype=np.float64
        )

        lgb_class_mae = np.mean(
            np.abs(actual_class - lgb_class_prediction)
        )
        naive_class_mae = np.mean(
            np.abs(actual_class - naive_class_prediction)
        )

        LGB_NAIVE_CLASS_COMPARISON.append({
            "fold": fold_number,
            "demand_class": demand_class,
            "rows": len(group),
            "LightGBM_MAE": lgb_class_mae,
            "Seasonal_Naive_MAE": naive_class_mae,
            "MAE_improvement_percent": (
                (naive_class_mae - lgb_class_mae)
                / naive_class_mae * 100
                if naive_class_mae > 0 else np.nan
            )
        })

    print("Matched rows:", len(comparison_df))
    print("Actual targets match:", np.array_equal(actual_lgb, actual_naive))

lgb_naive_comparison_df = pd.DataFrame(LGB_NAIVE_COMPARISON)

lgb_naive_class_comparison_df = pd.DataFrame(
    LGB_NAIVE_CLASS_COMPARISON
)

print("\nOverall LightGBM vs Seasonal Naive")
display(lgb_naive_comparison_df.round(6))

print("\nComparison by demand class")
display(
    lgb_naive_class_comparison_df
    .sort_values(["fold", "demand_class"])
    .round(6)
    .reset_index(drop=True)
)


Auditing Fold 1
Matched rows: 56000
Actual targets match: True

Auditing Fold 2
Matched rows: 56000
Actual targets match: True

Auditing Fold 3
Matched rows: 56000
Actual targets match: True

Overall LightGBM vs Seasonal Naive


,fold,rows,LightGBM_MAE,Seasonal_Naive_MAE,MAE_improvement_percent,LightGBM_RMSE,Seasonal_Naive_RMSE,RMSE_improvement_percent,LightGBM_WAPE_percent,Seasonal_Naive_WAPE_percent,WAPE_improvement_percent
0,1,56000,0.903292,1.202036,24.853108,2.709361,3.457988,21.649207,72.735138,96.790613,24.853108
1,2,56000,0.931940,1.168768,20.263019,2.403668,2.749120,12.565899,69.934541,87.706533,20.263019
2,3,56000,0.892580,1.153179,22.598283,2.251032,2.614144,13.890290,67.696626,87.461401,22.598283



Comparison by demand class


,fold,demand_class,rows,LightGBM_MAE,Seasonal_Naive_MAE,MAE_improvement_percent
0,1,erratic,1624,3.385730,5.144704,34.190006
1,1,intermittent,40320,0.580036,0.769767,24.647871
2,1,lumpy,10024,1.255280,1.619713,22.499806
3,1,no_demand,448,0.122768,0.122768,0.000000
4,1,smooth,3584,2.528176,3.245257,22.096274
5,2,erratic,1624,3.513471,4.229064,16.920834
6,2,intermittent,40320,0.575466,0.728274,20.982204
7,2,lumpy,10024,1.387226,1.777634,21.962220
8,2,no_demand,448,0.705933,0.801339,11.905889
9,2,smooth,3584,2.527394,3.080636,17.958712


In [47]:
import numpy as np
import pandas as pd

FOLD_EVALUATION_TYPES = {
    1: "development_fold_1",
    2: "development_fold_2",
    3: "development_fold_3"
}

comparison_keys = ["id", "origin_day_num", "target_day_num"]

horizon_comparison_results = []

for fold_number, evaluation_type in FOLD_EVALUATION_TYPES.items():

    print(f"Processing Fold {fold_number}...")

    lgb_df = LGB_VALIDATION_PREDICTIONS[fold_number].copy()

    naive_df = seasonal_naive_df.loc[
        seasonal_naive_df["evaluation_type"] == evaluation_type,
        comparison_keys + ["target_sales", "naive_forecast"]
    ].copy()

    lgb_df["horizon"] = (
        lgb_df["target_day_num"] - lgb_df["origin_day_num"]
    )

    naive_df["horizon"] = (
        naive_df["target_day_num"] - naive_df["origin_day_num"]
    )

    merged_df = lgb_df.merge(
        naive_df,
        on=comparison_keys + ["horizon"],
        how="inner",
        suffixes=("_lgb", "_naive"),
        validate="one_to_one"
    )

    assert len(merged_df) == 2000 * 28, (
        f"Unexpected number of rows in Fold {fold_number}"
    )

    assert np.array_equal(
        merged_df["target_sales_lgb"].to_numpy(),
        merged_df["target_sales_naive"].to_numpy()
    ), f"Actual target mismatch in Fold {fold_number}"

    for horizon, group in merged_df.groupby("horizon"):

        actual = group["target_sales_lgb"].to_numpy()
        lgb_pred = group["prediction_p50"].to_numpy()
        naive_pred = group["naive_forecast"].to_numpy()

        lgb_error = lgb_pred - actual
        naive_error = naive_pred - actual

        lgb_mae = np.mean(np.abs(lgb_error))
        naive_mae = np.mean(np.abs(naive_error))

        lgb_rmse = np.sqrt(np.mean(lgb_error ** 2))
        naive_rmse = np.sqrt(np.mean(naive_error ** 2))

        actual_total = np.sum(actual)

        lgb_wape = (
            np.sum(np.abs(lgb_error)) / actual_total * 100
            if actual_total > 0 else np.nan
        )

        naive_wape = (
            np.sum(np.abs(naive_error)) / actual_total * 100
            if actual_total > 0 else np.nan
        )

        horizon_comparison_results.append({
            "fold": fold_number,
            "horizon": horizon,
            "rows": len(group),
            "LightGBM_MAE": lgb_mae,
            "Seasonal_Naive_MAE": naive_mae,
            "MAE_improvement_percent": (
                (naive_mae - lgb_mae) / naive_mae * 100
                if naive_mae > 0 else np.nan
            ),
            "LightGBM_RMSE": lgb_rmse,
            "Seasonal_Naive_RMSE": naive_rmse,
            "RMSE_improvement_percent": (
                (naive_rmse - lgb_rmse) / naive_rmse * 100
                if naive_rmse > 0 else np.nan
            ),
            "LightGBM_WAPE_percent": lgb_wape,
            "Seasonal_Naive_WAPE_percent": naive_wape,
            "WAPE_improvement_percent": (
                (naive_wape - lgb_wape) / naive_wape * 100
                if naive_wape > 0 else np.nan
            )
        })

horizon_comparison_df = pd.DataFrame(horizon_comparison_results)

horizon_comparison_df = horizon_comparison_df.sort_values(
    ["fold", "horizon"]
).reset_index(drop=True)

print("\nHorizon-level comparison:")
display(
    horizon_comparison_df.round(4)
)

print("\nVerification:")
print("Total comparison rows:", len(horizon_comparison_df))
print("Expected rows:", 3 * 28)
print("All horizons present:",
      all(
          set(group["horizon"]) == set(range(1, 29))
          for _, group in horizon_comparison_df.groupby("fold")
      ))
print("All horizon groups contain 2,000 observations:",
      (horizon_comparison_df["rows"] == 2000).all())

Processing Fold 1...
Processing Fold 2...
Processing Fold 3...

Horizon-level comparison:


,fold,horizon,rows,LightGBM_MAE,Seasonal_Naive_MAE,MAE_improvement_percent,LightGBM_RMSE,Seasonal_Naive_RMSE,RMSE_improvement_percent,LightGBM_WAPE_percent,Seasonal_Naive_WAPE_percent,WAPE_improvement_percent
0,1,1,2000,0.7689,0.9770,21.3023,1.7374,2.1480,19.1142,68.5578,87.1155,21.3023
1,1,2,2000,0.7817,1.0560,25.9790,1.8615,2.4199,23.0739,67.9411,91.7862,25.9790
2,1,3,2000,0.6648,0.9740,31.7408,1.4857,2.2705,34.5650,68.5760,100.4642,31.7408
3,1,4,2000,0.8120,1.3405,39.4287,3.1412,5.2219,39.8450,73.4140,121.2025,39.4287
4,1,5,2000,0.8725,1.1175,21.9196,3.2816,3.6370,9.7710,71.2575,91.2617,21.9196
...,...,...,...,...,...,...,...,...,...,...,...,...
79,3,24,2000,0.7630,1.0560,27.7463,1.8953,2.5607,25.9855,73.1193,101.1979,27.7463
80,3,25,2000,0.7618,1.0220,25.4629,1.8623,2.2834,18.4412,74.7933,100.3436,25.4629
81,3,26,2000,0.9844,1.1765,16.3288,2.3045,2.5418,9.3339,70.0884,83.7665,16.3288
82,3,27,2000,1.1599,1.3765,15.7364,3.1997,3.0930,-3.4497,68.0686,80.7805,15.7364



Verification:
Total comparison rows: 84
Expected rows: 84
All horizons present: True
All horizon groups contain 2,000 observations: True


#### Aggregate performance across the three folds

In [48]:
horizon_summary_df = (
    horizon_comparison_df
    .groupby("horizon")
    .agg(
        mean_lgb_mae=("LightGBM_MAE", "mean"),
        mean_naive_mae=("Seasonal_Naive_MAE", "mean"),
        mean_mae_improvement=("MAE_improvement_percent", "mean"),
        mean_lgb_rmse=("LightGBM_RMSE", "mean"),
        mean_naive_rmse=("Seasonal_Naive_RMSE", "mean"),
        mean_rmse_improvement=("RMSE_improvement_percent", "mean"),
        mean_wape_improvement=("WAPE_improvement_percent", "mean"),
        folds_lgb_better_mae=(
            "MAE_improvement_percent",
            lambda x: (x > 0).sum()
        ),
        folds_lgb_better_rmse=(
            "RMSE_improvement_percent",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

print("Average performance by forecast horizon")
display(horizon_summary_df.round(4))

print("\nHorizons where LightGBM did not outperform Seasonal Naive on MAE:")
display(
    horizon_summary_df[
        horizon_summary_df["folds_lgb_better_mae"] < 3
    ].round(4)
)

print("\nHorizons where LightGBM did not outperform Seasonal Naive on RMSE:")
display(
    horizon_summary_df[
        horizon_summary_df["folds_lgb_better_rmse"] < 3
    ].round(4)
)

Average performance by forecast horizon


,horizon,mean_lgb_mae,mean_naive_mae,mean_mae_improvement,mean_lgb_rmse,mean_naive_rmse,mean_rmse_improvement,mean_wape_improvement,folds_lgb_better_mae,folds_lgb_better_rmse
0,1,0.8271,1.0575,21.8044,1.9137,2.3737,19.6152,21.8044,3,3
1,2,0.7975,1.0435,23.5657,2.0238,2.4322,16.9705,23.5657,3,3
2,3,0.7630,1.0202,25.3743,1.7797,2.3968,26.0510,25.3743,3,3
3,4,0.8349,1.1752,28.1848,2.5353,3.4787,22.9211,28.1848,3,3
4,5,0.8533,1.0672,19.9974,2.3729,2.6825,11.9421,19.9974,3,3
5,6,1.1246,1.3320,15.5966,3.2270,3.2250,-0.1272,15.5966,3,1
6,7,1.1025,1.3038,15.4792,3.1315,3.1707,0.4113,15.4792,3,2
7,8,0.8495,1.0955,22.3379,2.2403,2.6861,16.5145,22.3379,3,3
8,9,0.8472,1.0955,22.6223,2.4069,2.8053,14.2806,22.6223,3,3
9,10,0.8247,1.0973,24.8548,2.2512,2.7315,17.6304,24.8548,3,3



Horizons where LightGBM did not outperform Seasonal Naive on MAE:


,horizon,mean_lgb_mae,mean_naive_mae,mean_mae_improvement,mean_lgb_rmse,mean_naive_rmse,mean_rmse_improvement,mean_wape_improvement,folds_lgb_better_mae,folds_lgb_better_rmse



Horizons where LightGBM did not outperform Seasonal Naive on RMSE:


,horizon,mean_lgb_mae,mean_naive_mae,mean_mae_improvement,mean_lgb_rmse,mean_naive_rmse,mean_rmse_improvement,mean_wape_improvement,folds_lgb_better_mae,folds_lgb_better_rmse
5,6,1.1246,1.3320,15.5966,3.2270,3.2250,-0.1272,15.5966,3,1
6,7,1.1025,1.3038,15.4792,3.1315,3.1707,0.4113,15.4792,3,2
26,27,1.1018,1.3670,19.3673,2.9564,3.2455,9.0188,19.3673,3,2


#### investigate the weaker horizons

In [49]:
import numpy as np
import pandas as pd

comparison_keys = ["id", "origin_day_num", "target_day_num"]

FOLD_EVALUATION_TYPES = {
    1: "development_fold_1",
    2: "development_fold_2",
    3: "development_fold_3"
}

diagnostic_horizons = [6, 7, 27]

diagnostic_results = []

for fold_number, evaluation_type in FOLD_EVALUATION_TYPES.items():

    lgb_df = LGB_VALIDATION_PREDICTIONS[fold_number].copy()

    naive_df = seasonal_naive_df[
        seasonal_naive_df["evaluation_type"] == evaluation_type
    ][
        comparison_keys + ["target_sales", "naive_forecast"]
    ].copy()

    merged_df = lgb_df.merge(
        naive_df,
        on=comparison_keys,
        how="inner",
        suffixes=("_lgb", "_naive"),
        validate="one_to_one"
    )

    merged_df["horizon"] = (
        merged_df["target_day_num"] -
        merged_df["origin_day_num"]
    )

    merged_df["lgb_absolute_error"] = (
        merged_df["target_sales_lgb"] -
        merged_df["prediction_p50"]
    ).abs()

    merged_df["naive_absolute_error"] = (
        merged_df["target_sales_naive"] -
        merged_df["naive_forecast"]
    ).abs()

    merged_df["lgb_squared_error"] = (
        merged_df["target_sales_lgb"] -
        merged_df["prediction_p50"]
    ) ** 2

    merged_df["naive_squared_error"] = (
        merged_df["target_sales_naive"] -
        merged_df["naive_forecast"]
    ) ** 2

    for horizon in diagnostic_horizons:

        horizon_df = merged_df[
            merged_df["horizon"] == horizon
        ]

        for model_name, error_column, squared_column in [
            ("LightGBM", "lgb_absolute_error", "lgb_squared_error"),
            ("Seasonal Naive", "naive_absolute_error", "naive_squared_error")
        ]:

            errors = horizon_df[error_column]

            diagnostic_results.append({
                "fold": fold_number,
                "horizon": horizon,
                "model": model_name,
                "rows": len(horizon_df),
                "MAE": errors.mean(),
                "RMSE": np.sqrt(horizon_df[squared_column].mean()),
                "median_absolute_error": errors.median(),
                "p95_absolute_error": errors.quantile(0.95),
                "max_absolute_error": errors.max()
            })

diagnostic_df = pd.DataFrame(diagnostic_results)

print("Error distribution for weaker forecast horizons")

display(
    diagnostic_df.sort_values(
        ["horizon", "fold", "model"]
    ).round(4)
)

Error distribution for weaker forecast horizons


,fold,horizon,model,rows,MAE,RMSE,median_absolute_error,p95_absolute_error,max_absolute_error
0,1,6,LightGBM,2000,1.1689,3.8770,0.3173,4.0000,111.0000
1,1,6,Seasonal Naive,2000,1.3555,3.8769,1.0000,5.0000,111.0000
6,2,6,LightGBM,2000,1.1635,3.0591,0.3197,4.1821,46.8055
7,2,6,Seasonal Naive,2000,1.3875,3.0939,1.0000,5.0000,40.0000
12,3,6,LightGBM,2000,1.0416,2.7448,0.3068,3.6702,51.3752
13,3,6,Seasonal Naive,2000,1.2530,2.7041,1.0000,5.0000,37.0000
2,1,7,LightGBM,2000,1.0968,3.5033,0.2655,4.1026,97.0000
3,1,7,Seasonal Naive,2000,1.3165,3.8504,1.0000,5.0000,97.0000
8,2,7,LightGBM,2000,1.0447,2.7110,0.2828,3.7360,38.8788
9,2,7,Seasonal Naive,2000,1.2615,2.7351,1.0000,5.0000,33.0000


### Model Fine-Tuning: LightGBM Optuna Tuning

#### Define Optuna objective

In [57]:
import gc
import copy
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from pathlib import Path

OPTUNA_FOLDS = [1, 2, 3]
OPTUNA_QUANTILE = 0.5
OPTUNA_MAX_BOOST_ROUND = 2000
OPTUNA_EARLY_STOPPING = 100

def load_lgb_fold_datasets(fold_number, params):
    train_path = LGB_DATASET_FILES[fold_number]["train"]
    validation_path = LGB_DATASET_FILES[fold_number]["validation"]

    required_columns = MODEL_FEATURES_FINAL + [TARGET_COLUMN]

    train_df = pd.read_parquet(
        train_path,
        columns=required_columns
    )

    validation_df = pd.read_parquet(
        validation_path,
        columns=required_columns
    )

    X_train = train_df[MODEL_FEATURES_FINAL]
    y_train = train_df[TARGET_COLUMN]

    X_validation = validation_df[MODEL_FEATURES_FINAL]
    y_validation = validation_df[TARGET_COLUMN]

    train_dataset = lgb.Dataset(
        X_train,
        label=y_train,
        feature_name=MODEL_FEATURES_FINAL,
        categorical_feature=LGB_CATEGORICAL_FEATURES,
        free_raw_data=True
    )

    train_dataset.construct()

    del train_df, X_train, y_train
    gc.collect()

    validation_dataset = lgb.Dataset(
        X_validation,
        label=y_validation,
        reference=train_dataset,
        feature_name=MODEL_FEATURES_FINAL,
        categorical_feature=LGB_CATEGORICAL_FEATURES,
        free_raw_data=True
    )

    validation_dataset.construct()

    del validation_df, X_validation, y_validation
    gc.collect()

    return train_dataset, validation_dataset


def suggest_lgb_params(trial):
    max_depth = trial.suggest_categorical(
        "max_depth",
        [-1, 6, 8, 10, 12]
    )

    if max_depth == -1:
        max_num_leaves = 63
    else:
        max_num_leaves = min(63, 2 ** max_depth)

    num_leaves = trial.suggest_int(
        "num_leaves",
        15,
        max_num_leaves
    )

    return {
        "objective": "quantile",
        "alpha": OPTUNA_QUANTILE,
        "metric": "quantile",
        "verbosity": -1,
        "seed": 42,
        "feature_fraction_seed": 42,
        "bagging_seed": 42,
        "num_threads": -1,
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.08, log=True
        ),
        "num_leaves": num_leaves,
        "max_depth": max_depth,
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 50, 500, log=True
        ),
        "feature_fraction": trial.suggest_float(
            "feature_fraction", 0.7, 1.0
        ),
        "bagging_fraction": trial.suggest_float(
            "bagging_fraction", 0.7, 1.0
        ),
        "bagging_freq": trial.suggest_int(
            "bagging_freq", 1, 7
        ),
        "lambda_l1": trial.suggest_float(
            "lambda_l1", 1e-8, 10.0, log=True
        ),
        "lambda_l2": trial.suggest_float(
            "lambda_l2", 1e-8, 10.0, log=True
        )
    }


def lgb_p50_objective(trial):
    params = suggest_lgb_params(trial)

    fold_losses = {}
    fold_iterations = {}

    for fold_number in OPTUNA_FOLDS:
        start_time = time.time()

        print(
            f"\nTrial {trial.number} | "
            f"Fold {fold_number} | P50"
        )

        train_dataset, validation_dataset = (
            load_lgb_fold_datasets(fold_number, params)
        )

        model = lgb.train(
            params=params,
            train_set=train_dataset,
            num_boost_round=OPTUNA_MAX_BOOST_ROUND,
            valid_sets=[validation_dataset],
            valid_names=["validation"],
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=OPTUNA_EARLY_STOPPING,
                    first_metric_only=True,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        fold_loss = model.best_score["validation"]["quantile"]
        best_iteration = model.best_iteration

        fold_losses[fold_number] = fold_loss
        fold_iterations[fold_number] = best_iteration

        elapsed = time.time() - start_time

        print(
            f"Fold {fold_number} | "
            f"Loss: {fold_loss:.6f} | "
            f"Best iteration: {best_iteration} | "
            f"Time: {elapsed / 60:.2f} min"
        )

        del model, train_dataset, validation_dataset
        gc.collect()

    mean_loss = float(np.mean(list(fold_losses.values())))

    trial.set_user_attr("fold_losses", fold_losses)
    trial.set_user_attr("fold_best_iterations", fold_iterations)

    print(f"\nTrial {trial.number} mean loss: {mean_loss:.6f}")

    return mean_loss


print("Optuna P50 objective defined.")
print("Development folds:", OPTUNA_FOLDS)
print("Objective: Mean validation quantile loss")
print("Maximum boosting rounds:", OPTUNA_MAX_BOOST_ROUND)

Optuna P50 objective defined.
Development folds: [1, 2, 3]
Objective: Mean validation quantile loss
Maximum boosting rounds: 2000


In [ ]:
OPTUNA_PILOT_TRIALS = 5

LGB_P50_STUDY = optuna.create_study(
    study_name="lightgbm_p50_development",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

LGB_P50_STUDY.optimize(
    lgb_p50_objective,
    n_trials=OPTUNA_PILOT_TRIALS,
    gc_after_trial=True,
    show_progress_bar=False
)

print("\nPilot completed.")

print("Completed trials:", len(LGB_P50_STUDY.trials))
print("Best mean validation loss:", LGB_P50_STUDY.best_value)
print("Best parameters:", LGB_P50_STUDY.best_params)
print("Best fold losses:", LGB_P50_STUDY.best_trial.user_attrs["fold_losses"])
print("Best fold iterations:", LGB_P50_STUDY.best_trial.user_attrs["fold_best_iterations"])

In [ ]:
# Cell 2: Run Optuna P50 optimisation

# Reproducibility
RANDOM_SEED = 42

OPTUNA_PILOT_TRIALS = 8

LGB_P50_STUDY = optuna.create_study(
    study_name="lightgbm_p50_development",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
)

LGB_P50_STUDY.optimize(
    lgb_p50_objective,
    n_trials=OPTUNA_PILOT_TRIALS,
    gc_after_trial=True,
    show_progress_bar=True
)

# Build trial summary
trial_records = []

for trial in LGB_P50_STUDY.trials:
    trial_records.append({
        "trial_number": trial.number,
        "state": trial.state.name,
        "mean_validation_loss": trial.value,
        "fold_1_loss": trial.user_attrs.get("fold_losses", {}).get(1),
        "fold_2_loss": trial.user_attrs.get("fold_losses", {}).get(2),
        "fold_3_loss": trial.user_attrs.get("fold_losses", {}).get(3),
        "fold_1_best_iteration": trial.user_attrs.get(
            "fold_best_iterations", {}
        ).get(1),
        "fold_2_best_iteration": trial.user_attrs.get(
            "fold_best_iterations", {}
        ).get(2),
        "fold_3_best_iteration": trial.user_attrs.get(
            "fold_best_iterations", {}
        ).get(3)
    })

trial_results_df = pd.DataFrame(trial_records)

trial_results_path = LGB_OUTPUT_DIR / "optuna_p50_trial_results.csv"
trial_results_df.to_csv(trial_results_path, index=False)

# Save best configuration and study details
best_trial = LGB_P50_STUDY.best_trial

best_result = {
    "study_name": "lightgbm_p50_development",
    "quantile": OPTUNA_QUANTILE,
    "development_folds": OPTUNA_FOLDS,
    "completed_trials": len([
        trial for trial in LGB_P50_STUDY.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
    ]),
    "best_trial_number": best_trial.number,
    "best_mean_validation_loss": LGB_P50_STUDY.best_value,
    "best_parameters": LGB_P50_STUDY.best_params,
    "best_fold_losses": best_trial.user_attrs["fold_losses"],
    "best_fold_iterations": best_trial.user_attrs[
        "fold_best_iterations"
    ]
}

best_result_path = LGB_OUTPUT_DIR / "optuna_p50_best_result.json"

with open(best_result_path, "w") as file:
    json.dump(best_result, file, indent=4)

# Display results
print("\nOPTUNA P50 OPTIMISATION COMPLETED")

print("\nCompleted trials:")
print(best_result["completed_trials"])

print("\nBest trial:")
print(best_trial.number)

print("\nBest mean validation quantile loss:")
print(f"{LGB_P50_STUDY.best_value:.6f}")

print("\nBest parameters:")
for parameter, value in LGB_P50_STUDY.best_params.items():
    print(f"{parameter}: {value}")

print("\nBest fold losses:")
for fold, loss in best_trial.user_attrs["fold_losses"].items():
    print(f"Fold {fold}: {loss:.6f}")

print("\nBest fold iterations:")
for fold, iteration in best_trial.user_attrs[
    "fold_best_iterations"
].items():
    print(f"Fold {fold}: {iteration}")

print("\nTrial results:")
display(trial_results_df)

print("\nSaved files:")
print(trial_results_path)
print(best_result_path)

[I 2026-11-04 03:32:42,135] A new study created in memory with name: lightgbm_p50_development


  0%|          | 0/8 [00:00<?, ?it/s]


Trial 0 | Fold 1 | P50
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
Fold 1 | Loss: 0.448206 | Best iteration: 698 | Time: 13.63 minutes

Trial 0 | Fold 2 | P50
Fold 2 | Loss: 0.463650 | Best iteration: 217 | Time: 6.07 minutes

Trial 0 | Fold 3 | P50
Fold 3 | Loss: 0.445864 | Best iteration: 197 | Time: 5.83 minutes

Trial 0 mean validation loss: 0.452573
[I 2026-11-04 03:58:14,582] Trial 0 finished with value: 0.4525734353049076 and parameters: {'learning_rate': 0.03361441955478077, 'num_leaves': 61, 'max_depth': -1, 'min_child_samples': 367, 'feature_fraction': 0.8803345035229626, 'bagging_fraction': 0.9124217733388136, 'bagging_freq': 1, 'lambda_l1': 5.360294728728285, 'lambda_l2': 0.31044435499483225}. Best is trial 0 with value: 0.4525734353049076.

Trial 1 | Fold 1 | P50
Fold 1 | Loss: 0.452564 |

,trial_number,state,mean_validation_loss,fold_1_loss,fold_2_loss,fold_3_loss,fold_1_best_iteration,fold_2_best_iteration,fold_3_best_iteration
0,0,COMPLETE,0.452573,0.448206,0.463650,0.445864,698,217,197
1,1,COMPLETE,0.455920,0.452564,0.466816,0.448379,945,378,463
2,2,COMPLETE,0.454600,0.451337,0.465092,0.447371,600,948,209
3,3,COMPLETE,0.456929,0.454084,0.467882,0.448821,564,293,282
4,4,COMPLETE,0.454071,0.450510,0.464065,0.447638,566,218,256
5,5,COMPLETE,0.457138,0.454340,0.467886,0.449187,495,253,270
6,6,COMPLETE,0.456994,0.454113,0.467897,0.448972,505,367,291
7,7,COMPLETE,0.453388,0.450046,0.462957,0.447162,1021,487,332



Saved files:
/kaggle/working/lightgbm_optuna/optuna_p50_trial_results.csv
/kaggle/working/lightgbm_optuna/optuna_p50_best_result.json


### Retrain LightGBM using the best hyperparameter

In [4]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import lightgbm as lgb

ROOT_DIR = Path(
    r"C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence"
)

CONFIG_PATH = (
    ROOT_DIR
    / "models"
    / "lightgbm"
    / "artifacts"
    / "lightgbm_model_config.json"
)

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    LGB_CONFIG = json.load(file)

MODEL_FEATURES_FINAL = LGB_CONFIG["model_features"]
NUMERICAL_FEATURES = LGB_CONFIG["numerical_features"]
CATEGORICAL_FEATURES = LGB_CONFIG["categorical_features"]
TARGET_COLUMN = LGB_CONFIG["target_column"]

assert len(MODEL_FEATURES_FINAL) == 32
assert len(NUMERICAL_FEATURES) == 24
assert len(CATEGORICAL_FEATURES) == 8
assert TARGET_COLUMN == "target_sales"

assert set(NUMERICAL_FEATURES).isdisjoint(
    CATEGORICAL_FEATURES
)

assert set(NUMERICAL_FEATURES + CATEGORICAL_FEATURES) == set(
    MODEL_FEATURES_FINAL
)

DATA_DIR = ROOT_DIR / "data" / "processed" / "lightgbm_corrected"

OUTPUT_DIR = (
    ROOT_DIR
    / "models"
    / "lightgbm"
    / "artifacts"
    / "development_training"
    / "tuned_p50"
)

MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

BEST_LGBM_PARAMS = {
    "objective": "quantile",
    "metric": "quantile",
    "alpha": 0.50,
    "learning_rate": 0.03361441955478077,
    "num_leaves": 61,
    "max_depth": -1,
    "min_child_samples": 367,
    "feature_fraction": 0.8803345035229626,
    "bagging_fraction": 0.9124217733388136,
    "bagging_freq": 1,
    "lambda_l1": 5.360294728728285,
    "lambda_l2": 0.31044435499483225,
    "verbosity": -1,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "data_random_seed": 42,
    "num_threads": -1
}

MAX_BOOST_ROUNDS = 3000
EARLY_STOPPING_ROUNDS = 100

print("Original LightGBM configuration loaded.")
print("Total features:", len(MODEL_FEATURES_FINAL))
print("Numerical features:", len(NUMERICAL_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Target:", TARGET_COLUMN)

print("\nAll numerical features:")
print(NUMERICAL_FEATURES)

print("\nAll categorical features:")
print(CATEGORICAL_FEATURES)

Original LightGBM configuration loaded.
Total features: 32
Numerical features: 24
Categorical features: 8
Target: target_sales

All numerical features:
['sales_at_origin', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_std_7', 'sales_rolling_mean_28', 'sales_rolling_std_28', 'sales_rolling_max_28', 'origin_sell_price', 'origin_price_missing', 'origin_price_change', 'origin_price_change_pct', 'origin_price_vs_8w_avg', 'origin_price_change_flag', 'target_month', 'target_year', 'target_is_weekend', 'target_day_of_month', 'target_week_of_year', 'target_is_event_day', 'target_is_snap_day', 'horizon']

All categorical features:
['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'target_weekday', 'target_event_name_1', 'target_event_type_1']


In [5]:
LGB_DATASET_FILES = {}

for fold in [1, 2, 3]:

    train_path = DATA_DIR / f"fold_{fold}_train.parquet"
    validation_path = DATA_DIR / f"fold_{fold}_validation.parquet"

    if not train_path.exists():
        raise FileNotFoundError(
            f"Missing corrected training dataset: {train_path}"
        )

    if not validation_path.exists():
        raise FileNotFoundError(
            f"Missing corrected validation dataset: {validation_path}"
        )

    LGB_DATASET_FILES[fold] = {
        "train": train_path,
        "validation": validation_path
    }

    print(f"\nFold {fold}")
    print("Training:", train_path)
    print("Validation:", validation_path)

print("\nAll three corrected fold datasets found.")


Fold 1
Training: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected\fold_1_train.parquet
Validation: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected\fold_1_validation.parquet

Fold 2
Training: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected\fold_2_train.parquet
Validation: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected\fold_2_validation.parquet

Fold 3
Training: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected\fold_3_train.parquet
Validation: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected\fold_3_validation.parquet

All three corrected fold datasets found.


In [8]:
LGB_TUNED_RESULTS = []

LGB_TUNED_MODELS = {}

for fold in [1, 2, 3]:

    print(f"\nTraining tuned LightGBM P50 — Fold {fold}")

    start_time = time.time()

    train_path = LGB_DATASET_FILES[fold]["train"]
    validation_path = LGB_DATASET_FILES[fold]["validation"]

    required_columns = list(
        dict.fromkeys(
            MODEL_FEATURES_FINAL + [TARGET_COLUMN]
        )
    )

    train_df = pd.read_parquet(
        train_path,
        columns=required_columns
    )

    validation_df = pd.read_parquet(
        validation_path,
        columns=required_columns
    )

    print("Training rows:", len(train_df))
    print("Validation rows:", len(validation_df))

    missing_train = [
        col for col in required_columns
        if col not in train_df.columns
    ]

    missing_validation = [
        col for col in required_columns
        if col not in validation_df.columns
    ]

    assert not missing_train, missing_train
    assert not missing_validation, missing_validation

    X_train = train_df[MODEL_FEATURES_FINAL].copy()
    y_train = train_df[TARGET_COLUMN].astype("float32")

    X_validation = validation_df[
        MODEL_FEATURES_FINAL
    ].copy()

    y_validation = validation_df[
        TARGET_COLUMN
    ].astype("float32")

    assert np.isfinite(y_train).all()
    assert np.isfinite(y_validation).all()

    assert (y_train >= 0).all()
    assert (y_validation >= 0).all()

    # Convert categorical features using training-set categories.
    for column in CATEGORICAL_FEATURES:

        train_categories = pd.Index(
            X_train[column].dropna().unique()
        )

        X_train[column] = pd.Categorical(
            X_train[column],
            categories=train_categories
        )

        X_validation[column] = pd.Categorical(
            X_validation[column],
            categories=train_categories
        )

    # Verify categorical data types.
    print("\nCategorical feature dtypes:")

    for column in CATEGORICAL_FEATURES:
        print(f"{column}: {X_train[column].dtype}")

        assert isinstance(
            X_train[column].dtype,
            pd.CategoricalDtype
        )

        assert isinstance(
            X_validation[column].dtype,
            pd.CategoricalDtype
        )

    print("\nAll categorical features converted successfully.")

    train_dataset = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=CATEGORICAL_FEATURES,
        free_raw_data=False
    )

    validation_dataset = lgb.Dataset(
        X_validation,
        label=y_validation,
        categorical_feature=CATEGORICAL_FEATURES,
        reference=train_dataset,
        free_raw_data=False
    )

    model = lgb.train(
        params=BEST_LGBM_PARAMS,
        train_set=train_dataset,
        num_boost_round=MAX_BOOST_ROUNDS,
        valid_sets=[validation_dataset],
        valid_names=["validation"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose=True
            ),
            lgb.log_evaluation(period=100)
        ]
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    best_iteration = model.best_iteration

    validation_loss = model.best_score[
        "validation"
    ]["quantile"]

    model_path = (
        MODEL_DIR
        / f"lightgbm_tuned_p50_fold_{fold}.txt"
    )

    model.save_model(
        str(model_path),
        num_iteration=best_iteration
    )

    LGB_TUNED_MODELS[fold] = model

    result = {
        "fold": fold,
        "quantile": 0.50,
        "best_iteration": best_iteration,
        "best_validation_pinball_loss": validation_loss,
        "training_rows": len(train_df),
        "validation_rows": len(validation_df),
        "training_time_minutes": elapsed_minutes,
        "checkpoint": str(model_path)
    }

    LGB_TUNED_RESULTS.append(result)

    print(f"\nFold {fold} completed.")
    print("Best iteration:", best_iteration)
    print("Validation pinball loss:", validation_loss)
    print(f"Training time: {elapsed_minutes:.2f} minutes")
    print("Saved:", model_path)

    del train_df, validation_df
    del X_train, X_validation
    del y_train, y_validation
    del train_dataset, validation_dataset


Training tuned LightGBM P50 — Fold 1
Training rows: 13776000
Validation rows: 56000

Categorical feature dtypes:
item_id: category
dept_id: category
cat_id: category
store_id: category
state_id: category
target_weekday: category
target_event_name_1: category
target_event_type_1: category

All categorical features converted successfully.
Training until validation scores don't improve for 100 rounds
[100]	validation's quantile: 0.453822
[200]	validation's quantile: 0.450094
[300]	validation's quantile: 0.44976
[400]	validation's quantile: 0.449531
[500]	validation's quantile: 0.449441
[600]	validation's quantile: 0.449414
[700]	validation's quantile: 0.449351
[800]	validation's quantile: 0.449309
[900]	validation's quantile: 0.449304
Early stopping, best iteration is:
[861]	validation's quantile: 0.449299

Fold 1 completed.
Best iteration: 861
Validation pinball loss: 0.44929912949403666
Training time: 9.63 minutes
Saved: C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intellig

In [9]:
for column in CATEGORICAL_FEATURES:LGB_TUNED_SUMMARY = pd.DataFrame(
    LGB_TUNED_RESULTS
)

summary_path = (
    RESULT_DIR
    / "tuned_lgbm_p50_training_summary.csv"
)

LGB_TUNED_SUMMARY.to_csv(
    summary_path,
    index=False
)

config = {
    "model": "LightGBM Quantile Regression",
    "quantile": 0.50,
    "tuning_study": "lightgbm_p50_development",
    "best_trial": 0,
    "best_optuna_validation_loss": 0.4525734353049076,
    "hyperparameters": BEST_LGBM_PARAMS,
    "max_boost_rounds": MAX_BOOST_ROUNDS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "features": MODEL_FEATURES_FINAL,
    "categorical_features": CATEGORICAL_FEATURES,
    "training_results": LGB_TUNED_RESULTS
}

config_path = (
    RESULT_DIR
    / "tuned_lgbm_p50_config.json"
)

with open(config_path, "w") as file:
    json.dump(config, file, indent=4)

print("Tuned LightGBM P50 training summary:")
display(LGB_TUNED_SUMMARY)

print("\nSaved:")
print(summary_path)
print(config_path)

Tuned LightGBM P50 training summary:


,fold,quantile,best_iteration,best_validation_pinball_loss,training_rows,validation_rows,training_time_minutes,checkpoint
0,1,0.5,861,0.449299,13776000,56000,9.628051,C:\Users\cjc72\Desktop\SalesForecasting\AI-Ret...
1,2,0.5,359,0.463928,14056000,56000,4.698938,C:\Users\cjc72\Desktop\SalesForecasting\AI-Ret...
2,3,0.5,246,0.445701,14336000,56000,4.015076,C:\Users\cjc72\Desktop\SalesForecasting\AI-Ret...



Saved:
C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\artifacts\development_training\tuned_p50\results\tuned_lgbm_p50_training_summary.csv
C:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\models\lightgbm\artifacts\development_training\tuned_p50\results\tuned_lgbm_p50_config.json


In [10]:
print("Verifying tuned LightGBM checkpoints...\n")

for fold in [1, 2, 3]:

    model_path = (
        MODEL_DIR
        / f"lightgbm_tuned_p50_fold_{fold}.txt"
    )

    assert model_path.exists(), (
        f"Missing model: {model_path}"
    )

    loaded_model = lgb.Booster(
        model_file=str(model_path)
    )

    assert loaded_model.num_feature() == len(
        MODEL_FEATURES_FINAL
    )

    print(
        f"Fold {fold}: PASSED | "
        f"Best iteration: {loaded_model.current_iteration()} | "
        f"Features: {loaded_model.num_feature()}"
    )

print("\nAll three tuned LightGBM models passed verification.")

Verifying tuned LightGBM checkpoints...

Fold 1: PASSED | Best iteration: 861 | Features: 32
Fold 2: PASSED | Best iteration: 359 | Features: 32
Fold 3: PASSED | Best iteration: 246 | Features: 32

All three tuned LightGBM models passed verification.
